In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 51.2 MB/s eta 0:00:00


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertTokenizer
import torch as pt
import re
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
from tqdm import tqdm
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy
import random
from collections import Counter

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, average_precision_score

from gensim.models import Word2Vec
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 1. Load and final procesisng of data
lemmatizer = WordNetLemmatizer()

#Lower case all text and lemmanise
def preprocess_text(text: str):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    tokens = [lemmatizer.lemmatize(tok) for tok in tokens]
    return tokens

def load_texts_for_word2vec(csv_path):
    """Load texts from CSV and tokenize (lowercase + lemmatize) for Word2Vec training"""
    print(f"Loading data from {csv_path}...")
    df = pd.read_csv(csv_path)
    df['combined_text'] = df['combined_text'].astype(str)

    # Tokenize with preprocessing
    tokenized_texts = [preprocess_text(text) for text in df['combined_text']]
    labels = df['label'].values

    return tokenized_texts, labels, df

# 2. Train Word2Vec model
def train_word2vec(tokenized_texts, vector_size=100, window=5, min_count=2, workers=4, epochs=10):
    """
    Train Word2Vec model

    Parameters:
    - vector_size: dimensionality of word vectors (100, 200, 300)
    - window: context window size
    - min_count: ignores words with frequency lower than this
    - workers: number of worker threads
    - epochs: number of training epochs
    """
    print("Training Word2Vec model...")
    model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        epochs=epochs,
        sg=1  # 1 for skip-gram, 0 for CBOW
    )
    print(f"Word2Vec model trained with vocabulary size: {len(model.wv)}")
    return model

# 3. Generate averaged Word2Vec features IN CHUNKS
def get_word2vec_features_chunked(tokenized_texts, w2v_model, labels, output_path, vector_size=100, chunk_size=1000):
    """Convert tokenized texts to averaged Word2Vec vectors and save in chunks"""

    # Create column names
    w2v_feature_cols = [f'w2v_{i}' for i in range(vector_size)]

    # Write header first
    header_df = pd.DataFrame(columns=w2v_feature_cols + ['label'])
    header_df.to_csv(output_path, index=False)

    # Process in chunks
    total_texts = len(tokenized_texts)
    num_chunks = (total_texts + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(range(num_chunks), desc="Processing chunks"):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, total_texts)

        # Process current chunk
        chunk_features = []
        for tokens in tokenized_texts[start_idx:end_idx]:
            # Get vectors for words in vocabulary
            valid_vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]

            if valid_vectors:
                # Average the vectors
                avg_vector = np.mean(valid_vectors, axis=0)
            else:
                # Zero vector if no words found
                avg_vector = np.zeros(vector_size)

            chunk_features.append(avg_vector)

        # Create DataFrame for this chunk
        chunk_array = np.array(chunk_features)
        chunk_df = pd.DataFrame(chunk_array, columns=w2v_feature_cols)
        chunk_df['label'] = labels[start_idx:end_idx]

        # Append to CSV (without header)
        chunk_df.to_csv(output_path, mode='a', header=False, index=False)

        # Clear memory
        del chunk_features, chunk_array, chunk_df

    print(f"Word2Vec features saved to {output_path}")

# 4. Complete pipeline with chunked saving
def create_word2vec_features(csv_path, output_path, vector_size=100, window=5, chunk_size=1000):
    """Complete pipeline to create Word2Vec features with memory-efficient saving"""

    # Load and tokenize
    tokenized_texts, labels, original_df = load_texts_for_word2vec(csv_path)

    # Train Word2Vec
    w2v_model = train_word2vec(
        tokenized_texts,
        vector_size=vector_size,
        window=window
    )

    # Generate and save features in chunks
    get_word2vec_features_chunked(
        tokenized_texts,
        w2v_model,
        labels,
        output_path,
        vector_size=vector_size,
        chunk_size=chunk_size
    )

    # Save the model
    model_path = output_path.replace('.csv', '.model')
    w2v_model.save(model_path)
    print(f"Word2Vec model saved to {model_path}")

    # Return model and first few rows for verification
    verification_df = pd.read_csv(output_path, nrows=5)

    return verification_df, w2v_model

In [5]:
# w2v_df, w2v_model = create_word2vec_features(
#     csv_path='final_datasets/Combined_Train.csv',
#     output_path='preprocessed_datasets/word2vec_features.csv',
#     vector_size=100,  # 100, 200, or 300
#     window=5,
#     chunk_size=100
# )

# print("\nWord2Vec features generated!")
# print(f"Shape: {w2v_df.shape}")
# w2v_df.head()

In [6]:
def scale_data(dataframe, oversample=False):
  x = dataframe[dataframe.columns[:-1]].values
  y = dataframe[dataframe.columns[-1]].values

  scaler = StandardScaler()
  x = scaler.fit_transform(x)

  if oversample:
    ros = RandomOverSampler()
    x, y = ros.fit_resample(x, y)

  data = np.hstack((x, np.reshape(y, (-1, 1))))

  return data, x, y

In [7]:
def split_dataset(df):
    train = df.sample(frac=0.8, random_state=42)
    test = df.drop(train.index)
    print(f"Total rows in dataset: {len(df)}")
    print(f"Total rows in train set: {len(train)}")
    print(f"Total rows in test set: {len(test)}")

    print("\nClass distribution in full dataset:")
    print(df['label'].value_counts())

    print("\nClass distribution in train set:")
    print(train['label'].value_counts())

    print("\nClass distribution in test set:")
    print(test['label'].value_counts())

    train, xtrain, ytrain = scale_data(train)
    test, xtest, ytest = scale_data(test)

    print("\n KNN \n")

    return train, xtrain, ytrain, test, xtest, ytest

In [8]:
def train_ml_models(xtrain, ytrain, xtest, ytest):
    print("\n KNN \n")
    knn_model = KNeighborsClassifier(n_neighbors=5)
    knn_model.fit(xtrain, ytrain)

    ypred = knn_model.predict(xtest)
    print(classification_report(ytest, ypred))

    print("\n Gaussian NB \n")
    nb_model = GaussianNB()
    nb_model = nb_model.fit(xtrain, ytrain)

    ypred = nb_model.predict(xtest)

    print(classification_report(ytest, ypred))

    print("\n Logistica Regression \n")
    lgr_model = LogisticRegression()
    lgr_model = lgr_model.fit(xtrain, ytrain)

    ypred = lgr_model.predict(xtest)

    print(classification_report(ytest, ypred))

    print("\n Support Vector Classifier \n")
    svc_model = SVC()
    svc_model = svc_model.fit(xtrain, ytrain)

    ypred = svc_model.predict(xtest)
    print(classification_report(ytest, ypred))

    print("\n XGBoost \n")
    xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
    xgb_model = xgb_model.fit(xtrain, ytrain)

    ypred = xgb_model.predict(xtest)
    print(classification_report(ytest, ypred))

    print("\n Random Forest Classifier \n")
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model.fit(xtrain, ytrain)

    ypred = rf_model.predict(xtest)
    print(classification_report(ytest, ypred))

In [9]:
class CNNModel(nn.Module):
    def __init__(self, embedding_dim, num_filters=100, filter_sizes=[3, 4, 5]):
        super(CNNModel, self).__init__()
        self.embedding_dim = embedding_dim

        # Reshape input to (batch_size, 1, embedding_dim) for conv1d
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(num_filters * len(filter_sizes), 64)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim)

        # Apply convolutions
        conv_outputs = []
        for conv in self.conv_layers:
            conv_out = self.relu(conv(x))  # (batch_size, num_filters, length)
            pooled = torch.max(conv_out, dim=2)[0]  # Max pooling
            conv_outputs.append(pooled)

        # Concatenate all conv outputs
        x = torch.cat(conv_outputs, dim=1)  # (batch_size, num_filters * len(filter_sizes))

        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x

In [10]:
class LSTMModel(nn.Module):
    def __init__(self, embedding_dim, hidden_dim=128, num_layers=2):
        super(LSTMModel, self).__init__()
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim,
                           num_layers=num_layers, batch_first=True, dropout=0.5)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim) - add sequence dimension

        # LSTM forward pass
        lstm_out, (hidden, cell) = self.lstm(x)  # hidden: (num_layers, batch_size, hidden_dim)

        # Use last hidden state
        x = hidden[-1]  # (batch_size, hidden_dim)

        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x

In [11]:
class CNNLSTMModel(nn.Module):
    def __init__(self, embedding_dim, num_filters=64, filter_sizes=[3, 4, 5],
                 hidden_dim=128, num_layers=1):
        super(CNNLSTMModel, self).__init__()
        self.embedding_dim = embedding_dim

        # CNN layers
        self.conv_layers = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])

        self.relu = nn.ReLU()

        # LSTM layers - input is concatenated conv outputs
        cnn_output_dim = num_filters * len(filter_sizes)
        self.lstm = nn.LSTM(input_size=cnn_output_dim, hidden_size=hidden_dim,
                           num_layers=num_layers, batch_first=True, dropout=0.5)

        self.dropout = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, embedding_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, embedding_dim)

        # Apply CNN
        conv_outputs = []
        for conv in self.conv_layers:
            conv_out = self.relu(conv(x))  # (batch_size, num_filters, length)
            pooled = torch.max(conv_out, dim=2)[0]  # Max pooling
            conv_outputs.append(pooled)

        x = torch.cat(conv_outputs, dim=1)  # (batch_size, cnn_output_dim)
        x = x.unsqueeze(1)  # (batch_size, 1, cnn_output_dim) - add sequence dimension

        # Apply LSTM
        lstm_out, (hidden, cell) = self.lstm(x)
        x = hidden[-1]  # (batch_size, hidden_dim)

        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc2(x))

        return x

In [12]:
def train_deep_learning_model(model, train_loader, test_loader, epochs=10, learning_rate=0.001, device='cpu'):
    """
    Train a PyTorch model and evaluate on test set.

    Parameters:
    - model: PyTorch model
    - train_loader: Training DataLoader
    - test_loader: Test DataLoader
    - epochs: Number of training epochs
    - learning_rate: Learning rate
    - device: 'cpu' or 'cuda'
    """

    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"Training on device: {device}")
    print(f"Model: {model.__class__.__name__}")
    print("=" * 50)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Evaluate on test set
        model.eval()
        test_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(device).float()
                batch_y = batch_y.to(device).float().unsqueeze(1)

                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                test_loss += loss.item()

                # Calculate accuracy
                predicted = (outputs > 0.5).float()
                correct += (predicted == batch_y).sum().item()
                total += batch_y.size(0)

        avg_train_loss = train_loss / len(train_loader)
        avg_test_loss = test_loss / len(test_loader)
        accuracy = correct / total

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Test Loss: {avg_test_loss:.4f} | Accuracy: {accuracy:.4f}")

    print("=" * 50)
    print(f"{model.__class__.__name__} training completed!")

    return model

# ==================== Wrapper Function ====================
def train_all_deep_learning_models(xtrain, ytrain, xtest, ytest, epochs=10, batch_size=32):
    """
    Train CNN, LSTM, and CNN-LSTM models.

    Parameters:
    - xtrain, ytrain: Training data and labels
    - xtest, ytest: Test data and labels
    - epochs: Number of training epochs
    - batch_size: Batch size for DataLoader
    """

    # Determine device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Convert to PyTorch tensors
    xtrain_tensor = torch.from_numpy(xtrain).float()
    ytrain_tensor = torch.from_numpy(ytrain).long()
    xtest_tensor = torch.from_numpy(xtest).float()
    ytest_tensor = torch.from_numpy(ytest).long()

    # Create DataLoaders
    train_dataset = TensorDataset(xtrain_tensor, ytrain_tensor)
    test_dataset = TensorDataset(xtest_tensor, ytest_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    embedding_dim = xtrain.shape[1]

    # Train CNN
    print("\n" + "=" * 50)
    print("TRAINING CNN MODEL")
    print("=" * 50)
    cnn_model = CNNModel(embedding_dim)
    cnn_model = train_deep_learning_model(cnn_model, train_loader, test_loader, epochs=epochs, device=device)

    # Train LSTM
    print("\n" + "=" * 50)
    print("TRAINING LSTM MODEL")
    print("=" * 50)
    lstm_model = LSTMModel(embedding_dim)
    lstm_model = train_deep_learning_model(lstm_model, train_loader, test_loader, epochs=epochs, device=device)

    # Train CNN-LSTM
    print("\n" + "=" * 50)
    print("TRAINING CNN-LSTM HYBRID MODEL")
    print("=" * 50)
    cnn_lstm_model = CNNLSTMModel(embedding_dim)
    cnn_lstm_model = train_deep_learning_model(cnn_lstm_model, train_loader, test_loader, epochs=epochs, device=device)

    return cnn_model, lstm_model, cnn_lstm_model

In [13]:
def train_deep_learning_model(model, train_loader, val_loader, test_loader, epochs=10, learning_rate=0.001, device='cpu'):
    """
    Train a PyTorch model with validation set and evaluate on test set.

    Parameters:
    - model: PyTorch model
    - train_loader: Training DataLoader
    - val_loader: Validation DataLoader
    - test_loader: Test DataLoader (NOT USED during training, only for final evaluation)
    - epochs: Number of training epochs
    - learning_rate: Learning rate
    - device: 'cpu' or 'cuda'
    """

    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    print(f"Training on device: {device}")
    print(f"Model: {model.__class__.__name__}")
    print("=" * 50)

    # Storage for history
    train_losses = []
    val_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # ==================== VALIDATION ====================
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device).float()
                batch_y = batch_y.to(device).float().unsqueeze(1)

                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

                # Calculate accuracy
                predicted = (outputs > 0.5).float()
                val_correct += (predicted == batch_y).sum().item()
                val_total += batch_y.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = val_correct / val_total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_accuracy:.4f}")

    print("=" * 50)
    print(f"{model.__class__.__name__} training completed!")

    # Test-set eval
    print("\n" + "=" * 50)
    print(f"FINAL EVALUATION ON TEST SET - {model.__class__.__name__}")
    print("=" * 50)

    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device).float()
            batch_y = batch_y.to(device).float().unsqueeze(1)

            outputs = model(batch_x)
            predicted = (outputs > 0.5).float()

            all_predictions.extend(predicted.cpu().numpy().flatten())
            all_labels.extend(batch_y.cpu().numpy().flatten())
            all_probabilities.extend(outputs.cpu().numpy().flatten())

    # Calculate metrics
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    all_probabilities = np.array(all_probabilities)

    test_accuracy = np.mean(all_predictions == all_labels)
    test_auc = roc_auc_score(all_labels, all_probabilities)

    print(f"\nTest Accuracy: {test_accuracy:.4f}")
    print(f"Test AUC Score: {test_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_predictions, target_names=['Fake (0)', 'Real (1)']))

    # Visualisation
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss curves
    axes[0, 0].plot(train_losses, label='Training Loss', marker='o')
    axes[0, 0].plot(val_losses, label='Validation Loss', marker='s')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title(f'{model.__class__.__name__} - Loss over Epochs')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Accuracy curve
    axes[0, 1].plot(val_accuracies, label='Validation Accuracy', marker='o', color='green')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title(f'{model.__class__.__name__} - Accuracy over Epochs')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0], cbar=False)
    axes[1, 0].set_xlabel('Predicted')
    axes[1, 0].set_ylabel('Actual')
    axes[1, 0].set_title(f'{model.__class__.__name__} - Confusion Matrix (Test Set)')
    axes[1, 0].set_xticklabels(['Fake (0)', 'Real (1)'])
    axes[1, 0].set_yticklabels(['Fake (0)', 'Real (1)'])

    # ROC Curve
    fpr, tpr, _ = roc_curve(all_labels, all_probabilities)
    axes[1, 1].plot(fpr, tpr, label=f'ROC Curve (AUC = {test_auc:.4f})', color='darkorange', lw=2)
    axes[1, 1].plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    axes[1, 1].set_xlabel('False Positive Rate')
    axes[1, 1].set_ylabel('True Positive Rate')
    axes[1, 1].set_title(f'{model.__class__.__name__} - ROC Curve (Test Set)')
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.show()

    return model, {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_accuracies': val_accuracies,
        'test_accuracy': test_accuracy,
        'test_auc': test_auc,
        'predictions': all_predictions,
        'labels': all_labels,
        'probabilities': all_probabilities
    }

In [14]:
def train_all_deep_learning_models(xtrain, ytrain, xtest, ytest, epochs=10, batch_size=32, val_split=0.2):
    """
    Train CNN, LSTM, and CNN-LSTM models with proper train/validation/test split.

    Parameters:
    - xtrain, ytrain: Training data and labels
    - xtest, ytest: Test data and labels (NOT used during training)
    - epochs: Number of training epochs
    - batch_size: Batch size for DataLoader
    - val_split: Fraction of training data to use for validation (default: 0.2)

    Verification:
    - Train set is partitioned into training and validation
    - Test set is NOT used during training, only for final evaluation
    """

    print("=" * 70)
    print("DATA SPLIT VERIFICATION")
    print("=" * 70)
    print(f"Training set size: {xtrain.shape[0]}")
    print(f"Test set size: {xtest.shape[0]}")
    print(f"Validation split: {val_split * 100}%")
    print(f"Effective training size: {int(xtrain.shape[0] * (1 - val_split))}")
    print(f"Effective validation size: {int(xtrain.shape[0] * val_split)}")
    print("✓ Test set will NOT be used during training phase")
    print("✓ Test set will ONLY be used for final evaluation")
    print("=" * 70 + "\n")

    # Determine device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Split training data into train and validation
    from sklearn.model_selection import train_test_split as sklearn_train_test_split
    xtrain_split, xval, ytrain_split, yval = sklearn_train_test_split(
        xtrain, ytrain, test_size=val_split, random_state=42, stratify=ytrain
    )

    # Convert to PyTorch tensors
    xtrain_tensor = torch.from_numpy(xtrain_split).float()
    ytrain_tensor = torch.from_numpy(ytrain_split).long()
    xval_tensor = torch.from_numpy(xval).float()
    yval_tensor = torch.from_numpy(yval).long()
    xtest_tensor = torch.from_numpy(xtest).float()
    ytest_tensor = torch.from_numpy(ytest).long()

    # Create DataLoaders
    train_dataset = TensorDataset(xtrain_tensor, ytrain_tensor)
    val_dataset = TensorDataset(xval_tensor, yval_tensor)
    test_dataset = TensorDataset(xtest_tensor, ytest_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    embedding_dim = xtrain.shape[1]

    models_history = {}

    # Train CNN
    print("\n" + "=" * 70)
    print("TRAINING CNN MODEL")
    print("=" * 70)
    cnn_model = CNNModel(embedding_dim)
    cnn_model, cnn_history = train_deep_learning_model(
        cnn_model, train_loader, val_loader, test_loader,
        epochs=epochs, device=device
    )
    models_history['CNN'] = cnn_history

    # Train LSTM
    print("\n" + "=" * 70)
    print("TRAINING LSTM MODEL")
    print("=" * 70)
    lstm_model = LSTMModel(embedding_dim)
    lstm_model, lstm_history = train_deep_learning_model(
        lstm_model, train_loader, val_loader, test_loader,
        epochs=epochs, device=device
    )
    models_history['LSTM'] = lstm_history

    # Train CNN-LSTM
    print("\n" + "=" * 70)
    print("TRAINING CNN-LSTM HYBRID MODEL")
    print("=" * 70)
    cnn_lstm_model = CNNLSTMModel(embedding_dim)
    cnn_lstm_model, cnn_lstm_history = train_deep_learning_model(
        cnn_lstm_model, train_loader, val_loader, test_loader,
        epochs=epochs, device=device
    )
    models_history['CNN-LSTM'] = cnn_lstm_history

    return cnn_model, lstm_model, cnn_lstm_model, models_history

In [15]:
def run_complete_pipeline_word2vec(
    dataset_csv_path,
    vector_size=300,
    window=5,
    chunk_size=1000,
    regenerate_features=True,
    train_ml=True,
    train_dl=True,
    dl_epochs=10,
    dl_batch_size=32,
    val_split=0.2,
    save_vectors=0,  # 0 = do not save vectors CSV, 1 = save
    w2v_output_path='word2vec_features.csv'
):
    """
    End-to-end pipeline for Word2Vec features → ML models → Deep learning models.
    - If regenerate_features is True (or file missing), trains Word2Vec and saves averaged features (when save_vectors==1).
    - Otherwise, loads the existing features CSV (when save_vectors==1).
    - When save_vectors==0, features are built in-memory and not written to disk.
    """
    print("\n" + "=" * 80)
    print("Starting Word2Vec + ML + DL pipeline")
    print("=" * 80 + "\n")

    # Step 1: Build or load Word2Vec features
    if save_vectors:
        if regenerate_features or not os.path.exists(w2v_output_path):
            print("Step 1: Training Word2Vec and creating features (saving to CSV)")
            print("-" * 80)
            _preview_df, _w2v_model = create_word2vec_features(
                csv_path=dataset_csv_path,
                output_path=w2v_output_path,
                vector_size=vector_size,
                window=window,
                chunk_size=chunk_size
            )
            feature_df = pd.read_csv(w2v_output_path)
        else:
            print("Step 1: Loading existing Word2Vec features")
            print("-" * 80)
            feature_df = pd.read_csv(w2v_output_path)
            print(f"Loaded features from {w2v_output_path}")
    else:
        print("Step 1: Training Word2Vec and creating features (in-memory, not saving CSV)")
        print("-" * 80)
        tokenized_texts, labels, _ = load_texts_for_word2vec(dataset_csv_path)
        w2v_model = train_word2vec(
            tokenized_texts,
            vector_size=vector_size,
            window=window
        )
        w2v_feature_cols = [f"w2v_{i}" for i in range(vector_size)]
        features = []
        for tokens in tokenized_texts:
            valid_vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
            if valid_vectors:
                features.append(np.mean(valid_vectors, axis=0))
            else:
                features.append(np.zeros(vector_size))
        feature_df = pd.DataFrame(np.array(features), columns=w2v_feature_cols)
        feature_df['label'] = labels
        print("Features built in memory (not saved).")

    # Step 2: Split dataset
    print("\nStep 2: Splitting dataset (80/20 train/test)")
    print("-" * 80)
    train, xtrain, ytrain, test, xtest, ytest = split_dataset(feature_df)

    # Step 3: Train ML models
    if train_ml:
        print("\nStep 3: Training traditional ML models")
        print("-" * 80)
        train_ml_models(xtrain, ytrain, xtest, ytest)
    else:
        print("\nStep 3: Skipping traditional ML models (train_ml=False)")

    # Step 4: Train deep learning models
    models_history = {}
    if train_dl:
        print("\nStep 4: Training deep learning models with validation/test split")
        print("-" * 80)
        cnn_model, lstm_model, cnn_lstm_model, models_history = train_all_deep_learning_models(
            xtrain, ytrain, xtest, ytest,
            epochs=dl_epochs,
            batch_size=dl_batch_size,
            val_split=val_split
        )
    else:
        print("\nStep 4: Skipping deep learning models (train_dl=False)")

    # Summary
    print("\n" + "=" * 80)
    print("Pipeline completed successfully")
    print("=" * 80)
    print(f"Word2Vec feature size: {xtrain.shape[1]} dimensions")
    print(f"Training samples: {len(xtrain)}")
    print(f"Test samples: {len(xtest)}")
    if train_ml:
        print("ML models trained (KNN, Naive Bayes, Logistic Regression, SVC, XGBoost, Random Forest)")
    if train_dl:
        print("DL models trained (CNN, LSTM, CNN-LSTM)")
        print(f"  - CNN test accuracy: {models_history['CNN']['test_accuracy']:.4f}")
        print(f"  - LSTM test accuracy: {models_history['LSTM']['test_accuracy']:.4f}")
        print(f"  - CNN-LSTM test accuracy: {models_history['CNN-LSTM']['test_accuracy']:.4f}")
    print("=" * 80 + "\n")

    return feature_df, xtrain, ytrain, xtest, ytest, models_history

In [16]:
# Averaged Word2Vec features for ML + sequence inputs for DL
def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def create_stratified_splits(df, label_col='label', test_size=0.15, val_size=0.0, seed=42):
    y = df[label_col].values
    all_indices = np.arange(len(df))

    train_idx, test_idx = train_test_split(
        all_indices,
        test_size=test_size,
        random_state=seed,
        stratify=y
    )

    if val_size and val_size > 0:
        y_train = y[train_idx]
        train_idx, val_idx = train_test_split(
            train_idx,
            test_size=val_size,
            random_state=seed,
            stratify=y_train
        )
    else:
        val_idx = np.array([], dtype=int)

    split_info = {
        'train_idx': np.asarray(train_idx, dtype=int),
        'val_idx': np.asarray(val_idx, dtype=int),
        'test_idx': np.asarray(test_idx, dtype=int)
    }

    print('Split summary (stratified):')
    if len(val_idx) > 0:
        print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
    else:
        print(f"Train: {len(train_idx)} | Test: {len(test_idx)}")

    # Safety checks to avoid leakage across partitions.
    train_set = set(split_info['train_idx'].tolist())
    val_set = set(split_info['val_idx'].tolist())
    test_set = set(split_info['test_idx'].tolist())
    if train_set.intersection(test_set):
        raise ValueError('Leakage detected: train and test indices overlap.')
    if val_set and train_set.intersection(val_set):
        raise ValueError('Leakage detected: train and val indices overlap.')
    if val_set and val_set.intersection(test_set):
        raise ValueError('Leakage detected: val and test indices overlap.')

    return split_info


def train_word2vec_model(tokenized_texts, vector_size=300, window=5, min_count=2, epochs=10, seed=42):
    model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=1,
        epochs=epochs,
        sg=1,
        seed=seed
    )
    return model


def average_word2vec_vector(tokens, w2v_model, vector_size):
    valid_vectors = [w2v_model.wv[tok] for tok in tokens if tok in w2v_model.wv]
    if not valid_vectors:
        return np.zeros(vector_size, dtype=np.float32)
    return np.mean(valid_vectors, axis=0).astype(np.float32)


def build_ml_matrices(tokenized_texts, labels, split_info, w2v_model, vector_size):
    features = np.array([
        average_word2vec_vector(tokens, w2v_model, vector_size)
        for tokens in tokenized_texts
    ], dtype=np.float32)

    y = np.array(labels)

    train_idx = split_info['train_idx']
    val_idx = split_info['val_idx']
    test_idx = split_info['test_idx']

    x_train = features[train_idx]
    y_train = y[train_idx]
    x_val = features[val_idx] if len(val_idx) > 0 else np.empty((0, features.shape[1]), dtype=np.float32)
    y_val = y[val_idx] if len(val_idx) > 0 else np.empty((0,), dtype=y.dtype)
    x_test = features[test_idx]
    y_test = y[test_idx]

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    if len(x_val) > 0:
        x_val = scaler.transform(x_val)
    x_test = scaler.transform(x_test)

    return {
        'x_train': x_train,
        'y_train': y_train,
        'x_val': x_val,
        'y_val': y_val,
        'x_test': x_test,
        'y_test': y_test,
        'scaler': scaler
    }


def evaluate_classifier(model, x, y):
    y_pred = model.predict(x)

    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(x)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_score = model.decision_function(x)
    else:
        y_score = y_pred

    precision, recall, f1, _ = precision_recall_fscore_support(y, y_pred, average='binary', zero_division=0)

    return {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc_score(y, y_score),
        'pr_auc': average_precision_score(y, y_score)
    }


def train_ml_models_v2(ml_data, seed=42):
    x_train, y_train = ml_data['x_train'], ml_data['y_train']
    x_test, y_test = ml_data['x_test'], ml_data['y_test']

    models = {
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'GaussianNB': GaussianNB(),
        'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=seed),
        'SVC': SVC(probability=True, class_weight='balanced', random_state=seed),
        'XGBoost': XGBClassifier(
            eval_metric='logloss',
            random_state=seed,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6
        ),
        'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=seed)
    }

    results = []
    fitted_models = {}

    for name, model in models.items():
        model.fit(x_train, y_train)
        fitted_models[name] = model
        metrics = evaluate_classifier(model, x_test, y_test)
        metrics['model'] = name
        results.append(metrics)

    result_df = pd.DataFrame(results).sort_values('f1', ascending=False)
    print('\nML results (test set):')
    print(result_df[['model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']])

    return fitted_models, result_df


def build_vocab(train_tokenized_texts, min_freq=2):
    counter = Counter(tok for tokens in train_tokenized_texts for tok in tokens)
    vocab = {'<PAD>': 0, '<UNK>': 1}

    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)

    return vocab


def encode_tokens(tokens, vocab, max_len):
    ids = [vocab.get(tok, vocab['<UNK>']) for tok in tokens]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids.extend([vocab['<PAD>']] * (max_len - len(ids)))
    return ids


def build_embedding_matrix(vocab, w2v_model, embedding_dim):
    matrix = np.random.normal(0, 0.1, (len(vocab), embedding_dim)).astype(np.float32)
    matrix[vocab['<PAD>']] = np.zeros(embedding_dim, dtype=np.float32)

    for token, idx in vocab.items():
        if token in ('<PAD>', '<UNK>'):
            continue
        if token in w2v_model.wv:
            matrix[idx] = w2v_model.wv[token]

    return matrix


def build_sequence_dataloaders(tokenized_texts, labels, split_info, vocab, max_len=200, batch_size=32):
    y = np.array(labels, dtype=np.float32)
    x = np.array([encode_tokens(tokens, vocab, max_len) for tokens in tokenized_texts], dtype=np.int64)

    train_idx = split_info['train_idx']
    val_idx = split_info['val_idx']
    test_idx = split_info['test_idx']

    x_train, y_train = x[train_idx], y[train_idx]
    x_test, y_test = x[test_idx], y[test_idx]

    if len(val_idx) == 0:
        # For non-CV paths, derive a validation subset from train indices.
        local_train_idx, local_val_idx = train_test_split(
            np.arange(len(x_train)),
            test_size=0.2,
            random_state=42,
            stratify=y_train
        )
        x_val, y_val = x_train[local_val_idx], y_train[local_val_idx]
        x_train, y_train = x_train[local_train_idx], y_train[local_train_idx]
    else:
        x_val, y_val = x[val_idx], y[val_idx]

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)),
        batch_size=batch_size,
        shuffle=False
    )
    test_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)),
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, val_loader, test_loader


class TextCNN(nn.Module):
    def __init__(self, embedding_matrix, num_filters=128, filter_sizes=(3, 4, 5), dropout=0.5):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=k)
            for k in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        emb = emb.transpose(1, 2)

        conv_outputs = []
        for conv in self.convs:
            out = torch.relu(conv(emb))
            pooled = torch.max(out, dim=2)[0]
            conv_outputs.append(pooled)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        return self.fc(x).squeeze(1)


class TextLSTM(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, num_layers=2, dropout=0.5):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        lstm_out, _ = self.lstm(emb)
        x = lstm_out[:, -1, :]
        x = self.dropout(x)
        return self.fc(x).squeeze(1)


class TextCNNLSTM(nn.Module):
    def __init__(self, embedding_matrix, num_filters=128, kernel_size=3, hidden_dim=128, dropout=0.5):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.conv = nn.Conv1d(embedding_dim, num_filters, kernel_size=kernel_size, padding=kernel_size // 2)
        self.lstm = nn.LSTM(
            input_size=num_filters,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        x = emb.transpose(1, 2)
        x = torch.relu(self.conv(x))
        x = x.transpose(1, 2)

        lstm_out, _ = self.lstm(x)
        x = self.dropout(lstm_out[:, -1, :])
        return self.fc(x).squeeze(1)


def _evaluate_dl(model, loader, criterion, device):
    model.eval()
    loss_sum = 0.0
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            loss_sum += loss.item()
            all_logits.extend(logits.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())

    all_logits = np.array(all_logits)
    all_labels = np.array(all_labels)
    all_probs = 1.0 / (1.0 + np.exp(-all_logits))
    all_preds = (all_probs >= 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )

    metrics = {
        'loss': loss_sum / max(1, len(loader)),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc_score(all_labels, all_probs),
        'pr_auc': average_precision_score(all_labels, all_probs),
        'labels': all_labels,
        'preds': all_preds,
        'probs': all_probs
    }

    return metrics


In [17]:
def train_ml_models_cv_v2(ml_data, seed=42, cv_folds=5):
    from sklearn.base import clone

    x_train, y_train = ml_data['x_train'], ml_data['y_train']
    x_test, y_test = ml_data['x_test'], ml_data['y_test']

    models = {
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'GaussianNB': GaussianNB(),
        'LogisticRegression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=seed),
        'SVC': SVC(probability=True, class_weight='balanced', random_state=seed),
        'XGBoost': XGBClassifier(
            eval_metric='logloss',
            random_state=seed,
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6
        ),
        'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=seed)
    }

    cv_splitter = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)

    selected_models = {}
    summary_rows = []
    fold_rows = []
    selected_test_rows = []

    for name, base_model in models.items():
        model_fold_rows = []
        best_fold_model = None
        best_fold_record = None

        for fold_no, (train_idx, val_idx) in enumerate(cv_splitter.split(x_train, y_train), start=1):
            fold_model = clone(base_model)
            fold_model.fit(x_train[train_idx], y_train[train_idx])

            val_metrics = evaluate_classifier(fold_model, x_train[val_idx], y_train[val_idx])
            test_metrics = evaluate_classifier(fold_model, x_test, y_test)

            record = {
                'model': name,
                'fold': fold_no,
                'val_accuracy': val_metrics['accuracy'],
                'val_precision': val_metrics['precision'],
                'val_recall': val_metrics['recall'],
                'val_f1': val_metrics['f1'],
                'val_roc_auc': val_metrics['roc_auc'],
                'val_pr_auc': val_metrics['pr_auc'],
                'test_accuracy': test_metrics['accuracy'],
                'test_precision': test_metrics['precision'],
                'test_recall': test_metrics['recall'],
                'test_f1': test_metrics['f1'],
                'test_roc_auc': test_metrics['roc_auc'],
                'test_pr_auc': test_metrics['pr_auc']
            }
            model_fold_rows.append(record)
            fold_rows.append(record)

            if best_fold_record is None:
                best_fold_record = record
                best_fold_model = fold_model
            else:
                better_val_f1 = record['val_f1'] > best_fold_record['val_f1']
                tie_val_f1 = np.isclose(record['val_f1'], best_fold_record['val_f1'])
                better_val_acc = record['val_accuracy'] > best_fold_record['val_accuracy']
                earlier_fold = record['fold'] < best_fold_record['fold']

                if better_val_f1 or (tie_val_f1 and (better_val_acc or (np.isclose(record['val_accuracy'], best_fold_record['val_accuracy']) and earlier_fold))):
                    best_fold_record = record
                    best_fold_model = fold_model

        fold_df_model = pd.DataFrame(model_fold_rows)

        summary_rows.append({
            'model': name,
            # Validation (CV) aggregates
            'cv_accuracy_mean': fold_df_model['val_accuracy'].mean(),
            'cv_accuracy_std': fold_df_model['val_accuracy'].std(ddof=0),
            'cv_precision_mean': fold_df_model['val_precision'].mean(),
            'cv_precision_std': fold_df_model['val_precision'].std(ddof=0),
            'cv_recall_mean': fold_df_model['val_recall'].mean(),
            'cv_recall_std': fold_df_model['val_recall'].std(ddof=0),
            'cv_f1_mean': fold_df_model['val_f1'].mean(),
            'cv_f1_std': fold_df_model['val_f1'].std(ddof=0),
            'cv_roc_auc_mean': fold_df_model['val_roc_auc'].mean(),
            'cv_roc_auc_std': fold_df_model['val_roc_auc'].std(ddof=0),
            'cv_pr_auc_mean': fold_df_model['val_pr_auc'].mean(),
            'cv_pr_auc_std': fold_df_model['val_pr_auc'].std(ddof=0),
            # Test aggregates across fold models
            'test_accuracy_mean': fold_df_model['test_accuracy'].mean(),
            'test_accuracy_std': fold_df_model['test_accuracy'].std(ddof=0),
            'test_precision_mean': fold_df_model['test_precision'].mean(),
            'test_precision_std': fold_df_model['test_precision'].std(ddof=0),
            'test_recall_mean': fold_df_model['test_recall'].mean(),
            'test_recall_std': fold_df_model['test_recall'].std(ddof=0),
            'test_f1_mean': fold_df_model['test_f1'].mean(),
            'test_f1_std': fold_df_model['test_f1'].std(ddof=0),
            'test_roc_auc_mean': fold_df_model['test_roc_auc'].mean(),
            'test_roc_auc_std': fold_df_model['test_roc_auc'].std(ddof=0),
            'test_pr_auc_mean': fold_df_model['test_pr_auc'].mean(),
            'test_pr_auc_std': fold_df_model['test_pr_auc'].std(ddof=0),
            # Selected best fold metadata (by validation F1)
            'selected_fold': int(best_fold_record['fold']),
            'selected_val_f1': best_fold_record['val_f1'],
            # Keep legacy holdout_* columns for downstream extractors
            'holdout_accuracy': best_fold_record['test_accuracy'],
            'holdout_precision': best_fold_record['test_precision'],
            'holdout_recall': best_fold_record['test_recall'],
            'holdout_f1': best_fold_record['test_f1'],
            'holdout_roc_auc': best_fold_record['test_roc_auc'],
            'holdout_pr_auc': best_fold_record['test_pr_auc']
        })

        selected_models[name] = best_fold_model
        selected_test_rows.append({
            'model': name,
            'selected_fold': int(best_fold_record['fold']),
            'accuracy': best_fold_record['test_accuracy'],
            'precision': best_fold_record['test_precision'],
            'recall': best_fold_record['test_recall'],
            'f1': best_fold_record['test_f1'],
            'roc_auc': best_fold_record['test_roc_auc'],
            'pr_auc': best_fold_record['test_pr_auc']
        })

    cv_df = pd.DataFrame(summary_rows).sort_values('cv_f1_mean', ascending=False).reset_index(drop=True)
    fold_df = pd.DataFrame(fold_rows).sort_values(['model', 'fold']).reset_index(drop=True)
    holdout_df = pd.DataFrame(selected_test_rows).sort_values('f1', ascending=False).reset_index(drop=True)

    ranked_df = cv_df.sort_values('cv_f1_mean', ascending=False).reset_index(drop=True)

    print(f"\nML results with {cv_folds}-fold CV (ranked by validation F1):")
    print(ranked_df[[
        'model', 'cv_f1_mean', 'cv_f1_std', 'test_f1_mean', 'test_f1_std', 'selected_fold', 'holdout_f1'
    ]])

    return selected_models, cv_df, holdout_df, ranked_df, fold_df

In [18]:
def train_dl_model_v2(model, train_loader, val_loader, test_loader, epochs=10, lr=1e-3, patience=3, device='cpu'):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    best_state = None
    best_epoch = -1
    early_stop_epoch = epochs
    wait = 0
    best_val_metrics = None

    history = {
        'train_loss': [],
        'val_loss': [],
        'val_f1': []
    }

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss = train_loss / max(1, len(train_loader))
        val_metrics = _evaluate_dl(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_f1'].append(val_metrics['f1'])

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val F1: {val_metrics['f1']:.4f}"
        )

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
            best_val_metrics = {
                'best_val_loss': val_metrics['loss'],
                'best_val_accuracy': val_metrics['accuracy'],
                'best_val_precision': val_metrics['precision'],
                'best_val_recall': val_metrics['recall'],
                'best_val_f1': val_metrics['f1'],
                'best_val_roc_auc': val_metrics['roc_auc'],
                'best_val_pr_auc': val_metrics['pr_auc']
            }
        else:
            wait += 1
            if wait >= patience:
                early_stop_epoch = epoch + 1
                print(f"Early stopping triggered at epoch {epoch + 1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    if best_val_metrics is None:
        fallback_val_metrics = _evaluate_dl(model, val_loader, criterion, device)
        best_val_metrics = {
            'best_val_loss': fallback_val_metrics['loss'],
            'best_val_accuracy': fallback_val_metrics['accuracy'],
            'best_val_precision': fallback_val_metrics['precision'],
            'best_val_recall': fallback_val_metrics['recall'],
            'best_val_f1': fallback_val_metrics['f1'],
            'best_val_roc_auc': fallback_val_metrics['roc_auc'],
            'best_val_pr_auc': fallback_val_metrics['pr_auc']
        }

    test_metrics = _evaluate_dl(model, test_loader, criterion, device)
    test_metrics['best_epoch'] = best_epoch
    test_metrics['early_stop_epoch'] = early_stop_epoch
    test_metrics.update(best_val_metrics)

    return model, history, test_metrics

def train_dl_models_v2(train_loader, val_loader, test_loader, embedding_matrix, epochs=10, lr=1e-3, patience=3):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    models = {
        'TextCNN': TextCNN(embedding_matrix),
        'TextLSTM': TextLSTM(embedding_matrix),
        'TextCNNLSTM': TextCNNLSTM(embedding_matrix)
    }

    results = []
    trained = {}

    for name, model in models.items():
        print(f"\nTraining {name} on sequence inputs...")
        trained_model, history, test_metrics = train_dl_model_v2(
            model,
            train_loader,
            val_loader,
            test_loader,
            epochs=epochs,
            lr=lr,
            patience=patience,
            device=device
        )
        trained[name] = {
            'model': trained_model,
            'history': history,
            'test_metrics': test_metrics
        }

        results.append({
            'model': name,
            'accuracy': test_metrics['accuracy'],
            'precision': test_metrics['precision'],
            'recall': test_metrics['recall'],
            'f1': test_metrics['f1'],
            'roc_auc': test_metrics['roc_auc'],
            'pr_auc': test_metrics['pr_auc'],
            'early_stop_epoch': test_metrics['early_stop_epoch']
        })

    results_df = pd.DataFrame(results).sort_values('f1', ascending=False)
    print('\nDL results (test set):')
    print(results_df)

    return trained, results_df

In [19]:
def _build_sequence_arrays(tokenized_texts, labels, vocab, max_len=200):
    y = np.array(labels, dtype=np.float32)
    x = np.array([encode_tokens(tokens, vocab, max_len) for tokens in tokenized_texts], dtype=np.int64)
    return x, y


def _build_fold_loaders_from_indices(x, y, train_idx, val_idx, batch_size=32):
    x_train, y_train = x[train_idx], y[train_idx]
    x_val, y_val = x[val_idx], y[val_idx]

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)),
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)),
        batch_size=batch_size,
        shuffle=False
    )
    return train_loader, val_loader


def train_dl_models_cv_v2(
    tokenized_texts,
    labels,
    split_info,
    vocab,
    embedding_matrix,
    cv_folds=5,
    max_len=200,
    batch_size=32,
    epochs=10,
    lr=1e-3,
    patience=3,
    seed=42
):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    x_all, y_all = _build_sequence_arrays(tokenized_texts, labels, vocab, max_len=max_len)
    holdout_train_idx = np.array(split_info['train_idx']).astype(int)
    holdout_test_idx = np.array(split_info['test_idx']).astype(int)

    if len(np.intersect1d(holdout_train_idx, holdout_test_idx)) > 0:
        raise ValueError('Leakage detected: holdout train and test indices overlap.')

    y_holdout = y_all[holdout_train_idx].astype(int)
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=seed)

    model_factories = {
        'TextCNN': lambda: TextCNN(embedding_matrix),
        'TextLSTM': lambda: TextLSTM(embedding_matrix),
        'TextCNNLSTM': lambda: TextCNNLSTM(embedding_matrix)
    }

    fixed_test_loader = DataLoader(
        TensorDataset(
            torch.from_numpy(x_all[holdout_test_idx]),
            torch.from_numpy(y_all[holdout_test_idx])
        ),
        batch_size=batch_size,
        shuffle=False
    )

    summary_rows = []
    fold_rows = []
    selected_test_rows = []
    selected_models = {}

    for name, model_factory in model_factories.items():
        print(f"\n{name}: {cv_folds}-fold CV on train partition")
        model_fold_records = []
        best_fold_record = None
        best_model_entry = None

        for fold_no, (rel_train_idx, rel_val_idx) in enumerate(skf.split(holdout_train_idx, y_holdout), start=1):
            abs_train_idx = holdout_train_idx[rel_train_idx]
            abs_val_idx = holdout_train_idx[rel_val_idx]

            fold_train_loader, fold_val_loader = _build_fold_loaders_from_indices(
                x_all,
                y_all,
                abs_train_idx,
                abs_val_idx,
                batch_size=batch_size
            )

            fold_model = model_factory()
            trained_model, fold_history, fold_metrics = train_dl_model_v2(
                fold_model,
                fold_train_loader,
                fold_val_loader,
                fixed_test_loader,
                epochs=epochs,
                lr=lr,
                patience=patience,
                device=device
            )

            fold_record = {
                'model': name,
                'fold': fold_no,
                'val_accuracy': fold_metrics['best_val_accuracy'],
                'val_precision': fold_metrics['best_val_precision'],
                'val_recall': fold_metrics['best_val_recall'],
                'val_f1': fold_metrics['best_val_f1'],
                'val_roc_auc': fold_metrics['best_val_roc_auc'],
                'val_pr_auc': fold_metrics['best_val_pr_auc'],
                'test_accuracy': fold_metrics['accuracy'],
                'test_precision': fold_metrics['precision'],
                'test_recall': fold_metrics['recall'],
                'test_f1': fold_metrics['f1'],
                'test_roc_auc': fold_metrics['roc_auc'],
                'test_pr_auc': fold_metrics['pr_auc'],
                'early_stop_epoch': fold_metrics['early_stop_epoch'],
                'best_epoch': fold_metrics['best_epoch']
            }
            model_fold_records.append(fold_record)
            fold_rows.append(fold_record)

            if best_fold_record is None:
                best_fold_record = fold_record
                best_model_entry = {
                    'model': trained_model,
                    'history': fold_history,
                    'test_metrics': fold_metrics,
                    'selected_fold': fold_no
                }
            else:
                better_val_f1 = fold_record['val_f1'] > best_fold_record['val_f1']
                tie_val_f1 = np.isclose(fold_record['val_f1'], best_fold_record['val_f1'])
                better_val_acc = fold_record['val_accuracy'] > best_fold_record['val_accuracy']
                earlier_fold = fold_record['fold'] < best_fold_record['fold']

                if better_val_f1 or (tie_val_f1 and (better_val_acc or (np.isclose(fold_record['val_accuracy'], best_fold_record['val_accuracy']) and earlier_fold))):
                    best_fold_record = fold_record
                    best_model_entry = {
                        'model': trained_model,
                        'history': fold_history,
                        'test_metrics': fold_metrics,
                        'selected_fold': fold_no
                    }

        fold_df_model = pd.DataFrame(model_fold_records)

        summary_rows.append({
            'model': name,
            # Validation (CV) aggregates
            'cv_accuracy_mean': fold_df_model['val_accuracy'].mean(),
            'cv_accuracy_std': fold_df_model['val_accuracy'].std(ddof=0),
            'cv_precision_mean': fold_df_model['val_precision'].mean(),
            'cv_precision_std': fold_df_model['val_precision'].std(ddof=0),
            'cv_recall_mean': fold_df_model['val_recall'].mean(),
            'cv_recall_std': fold_df_model['val_recall'].std(ddof=0),
            'cv_f1_mean': fold_df_model['val_f1'].mean(),
            'cv_f1_std': fold_df_model['val_f1'].std(ddof=0),
            'cv_roc_auc_mean': fold_df_model['val_roc_auc'].mean(),
            'cv_roc_auc_std': fold_df_model['val_roc_auc'].std(ddof=0),
            'cv_pr_auc_mean': fold_df_model['val_pr_auc'].mean(),
            'cv_pr_auc_std': fold_df_model['val_pr_auc'].std(ddof=0),
            # Test aggregates across fold models
            'test_accuracy_mean': fold_df_model['test_accuracy'].mean(),
            'test_accuracy_std': fold_df_model['test_accuracy'].std(ddof=0),
            'test_precision_mean': fold_df_model['test_precision'].mean(),
            'test_precision_std': fold_df_model['test_precision'].std(ddof=0),
            'test_recall_mean': fold_df_model['test_recall'].mean(),
            'test_recall_std': fold_df_model['test_recall'].std(ddof=0),
            'test_f1_mean': fold_df_model['test_f1'].mean(),
            'test_f1_std': fold_df_model['test_f1'].std(ddof=0),
            'test_roc_auc_mean': fold_df_model['test_roc_auc'].mean(),
            'test_roc_auc_std': fold_df_model['test_roc_auc'].std(ddof=0),
            'test_pr_auc_mean': fold_df_model['test_pr_auc'].mean(),
            'test_pr_auc_std': fold_df_model['test_pr_auc'].std(ddof=0),
            # Selected fold metadata (by validation F1)
            'selected_fold': int(best_fold_record['fold']),
            'selected_val_f1': best_fold_record['val_f1'],
            # Keep legacy holdout_* fields for extractors
            'holdout_accuracy': best_fold_record['test_accuracy'],
            'holdout_precision': best_fold_record['test_precision'],
            'holdout_recall': best_fold_record['test_recall'],
            'holdout_f1': best_fold_record['test_f1'],
            'holdout_roc_auc': best_fold_record['test_roc_auc'],
            'holdout_pr_auc': best_fold_record['test_pr_auc'],
            'holdout_early_stop_epoch': int(best_fold_record['early_stop_epoch'])
        })

        selected_models[name] = best_model_entry
        selected_test_rows.append({
            'model': name,
            'selected_fold': int(best_fold_record['fold']),
            'accuracy': best_fold_record['test_accuracy'],
            'precision': best_fold_record['test_precision'],
            'recall': best_fold_record['test_recall'],
            'f1': best_fold_record['test_f1'],
            'roc_auc': best_fold_record['test_roc_auc'],
            'pr_auc': best_fold_record['test_pr_auc'],
            'early_stop_epoch': int(best_fold_record['early_stop_epoch'])
        })

    cv_df = pd.DataFrame(summary_rows).sort_values('cv_f1_mean', ascending=False).reset_index(drop=True)
    holdout_df = pd.DataFrame(selected_test_rows).sort_values('f1', ascending=False).reset_index(drop=True)
    fold_df = pd.DataFrame(fold_rows).sort_values(['model', 'fold']).reset_index(drop=True)

    ranked_df = cv_df.sort_values('cv_f1_mean', ascending=False).reset_index(drop=True)

    print(f"\nDL results with {cv_folds}-fold CV (ranked by validation F1):")
    print(ranked_df[[
        'model', 'cv_f1_mean', 'cv_f1_std', 'test_f1_mean', 'test_f1_std', 'selected_fold', 'holdout_f1'
    ]])

    return selected_models, cv_df, holdout_df, ranked_df, fold_df

In [20]:
def run_hybrid_pipeline_v2(
    dataset_csv_path,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.15,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
):
    set_global_seed(seed)

    df = pd.read_csv(dataset_csv_path)
    df['combined_text'] = df['combined_text'].astype(str)

    tokenized_texts = [preprocess_text(text) for text in df['combined_text']]
    labels = df['label'].values

    split_info = create_stratified_splits(
        df,
        label_col='label',
        test_size=test_size,
        # In CV mode, keep the full train partition for folds and avoid a pre-carved val split.
        val_size=0.0 if enable_cv else val_size,
        seed=seed
    )

    if transductive_w2v:
        w2v_train_tokens = tokenized_texts
        regime = 'transductive'
    else:
        w2v_train_tokens = [tokenized_texts[i] for i in split_info['train_idx']]
        regime = 'strict'

    print(f"\nWord2Vec regime: {regime}")
    w2v_model = train_word2vec_model(
        w2v_train_tokens,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        epochs=w2v_epochs,
        seed=seed
    )

    ml_data = build_ml_matrices(tokenized_texts, labels, split_info, w2v_model, vector_size)
    if enable_cv:
        ml_models, ml_results_cv, ml_results_test, ml_results, ml_fold_results = train_ml_models_cv_v2(
            ml_data,
            seed=seed,
            cv_folds=cv_folds
        )
    else:
        ml_models, ml_results = train_ml_models_v2(ml_data, seed=seed)
        ml_results_cv = None
        ml_results_test = ml_results
        ml_fold_results = None

    train_tokens = [tokenized_texts[i] for i in split_info['train_idx']]
    vocab = build_vocab(train_tokens, min_freq=min_count)
    embedding_matrix = build_embedding_matrix(vocab, w2v_model, vector_size)

    if enable_cv:
        dl_models, dl_results_cv, dl_results_test, dl_results, dl_fold_results = train_dl_models_cv_v2(
            tokenized_texts=tokenized_texts,
            labels=labels,
            split_info=split_info,
            vocab=vocab,
            embedding_matrix=embedding_matrix,
            cv_folds=cv_folds,
            max_len=max_len,
            batch_size=dl_batch_size,
            epochs=dl_epochs,
            lr=dl_lr,
            patience=dl_patience,
            seed=seed
        )
    else:
        train_loader, val_loader, test_loader = build_sequence_dataloaders(
            tokenized_texts,
            labels,
            split_info,
            vocab,
            max_len=max_len,
            batch_size=dl_batch_size
        )

        dl_models, dl_results = train_dl_models_v2(
            train_loader,
            val_loader,
            test_loader,
            embedding_matrix,
            epochs=dl_epochs,
            lr=dl_lr,
            patience=dl_patience
        )
        dl_results_cv = None
        dl_results_test = dl_results
        dl_fold_results = None

    return {
        'split_info': split_info,
        'w2v_model': w2v_model,
        'word2vec_regime': regime,
        'vector_size': vector_size,
        'max_len': max_len,
        'ml_scaler': ml_data['scaler'],
        'ml_models': ml_models,
        'ml_results': ml_results,
        'ml_results_cv': ml_results_cv,
        'ml_results_test': ml_results_test,
        'ml_fold_results': ml_fold_results,
        'dl_models': dl_models,
        'dl_results': dl_results,
        'dl_results_cv': dl_results_cv,
        'dl_results_test': dl_results_test,
        'dl_fold_results': dl_fold_results,
        'vocab': vocab,
        'selection_policy': 'best_fold_by_validation_f1' if enable_cv else 'single_train_validation_split'
    }

In [21]:
def _prepare_cross_dataset_ml_matrix(dataset_path, w2v_model, scaler):
    df = pd.read_csv(dataset_path).dropna()
    df['combined_text'] = df['combined_text'].astype(str)
    tokenized_texts = [preprocess_text(text) for text in df['combined_text']]
    labels = np.asarray(df['label'].values, dtype=int)

    vector_size = w2v_model.vector_size
    features = np.array([
        average_word2vec_vector(tokens, w2v_model, vector_size)
        for tokens in tokenized_texts
    ], dtype=np.float32)

    x_scaled = scaler.transform(features)
    return tokenized_texts, x_scaled, labels


def _prepare_cross_dataset_dl_loader(tokenized_texts, labels, vocab, max_len=200, batch_size=64):
    x = np.array([encode_tokens(tokens, vocab, max_len) for tokens in tokenized_texts], dtype=np.int64)
    y = np.asarray(labels, dtype=np.float32)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
        batch_size=batch_size,
        shuffle=False
    )
    return loader


def _build_unseen_summary_table(model_name, model_type, y_true, y_pred, metrics, target_names=('Fake (0)', 'Real (1)')):
    report = classification_report(
        y_true,
        y_pred,
        target_names=list(target_names),
        output_dict=True,
        zero_division=0
    )

    rows = []
    for class_name in target_names:
        class_report = report[class_name]
        rows.append({
            'model_type': model_type,
            'model': model_name,
            'sub_row': class_name,
            'precision': class_report['precision'],
            'recall': class_report['recall'],
            'f1': class_report['f1-score'],
            'support': class_report['support'],
            'accuracy': np.nan,
            'roc_auc': np.nan,
            'pr_auc': np.nan
        })

    weighted_report = report['weighted avg']
    rows.append({
        'model_type': model_type,
        'model': model_name,
        'sub_row': 'Weighted Avg',
        'precision': weighted_report['precision'],
        'recall': weighted_report['recall'],
        'f1': weighted_report['f1-score'],
        'support': weighted_report['support'],
        'accuracy': np.nan,
        'roc_auc': np.nan,
        'pr_auc': np.nan
    })

    rows.append({
        'model_type': model_type,
        'model': model_name,
        'sub_row': 'Overall',
        'precision': weighted_report['precision'],
        'recall': weighted_report['recall'],
        'f1': weighted_report['f1-score'],
        'support': weighted_report['support'],
        'accuracy': metrics['accuracy'],
        'roc_auc': metrics['roc_auc'],
        'pr_auc': metrics['pr_auc']
    })

    table = pd.DataFrame(rows)
    table['sub_row'] = pd.Categorical(
        table['sub_row'],
        categories=['Fake (0)', 'Real (1)', 'Weighted Avg', 'Overall'],
        ordered=True
    )
    table = table.sort_values(['model_type', 'model', 'sub_row']).set_index(['model_type', 'model', 'sub_row'])
    return table[['precision', 'recall', 'f1', 'support', 'accuracy', 'roc_auc', 'pr_auc']]


def cross_dataset_evaluation_word2vec(result_bundle, current_dataset_name, all_datasets_paths, batch_size=64):
    print("\n" + "!" * 70)
    print(f"CROSS-DATASET GENERALIZATION: {current_dataset_name}")
    print("!" * 70)

    ml_models = result_bundle['ml_models']
    dl_models = result_bundle['dl_models']
    w2v_model = result_bundle['w2v_model']
    scaler = result_bundle['ml_scaler']
    vocab = result_bundle['vocab']
    max_len = result_bundle.get('max_len', 200)

    all_results = {}

    for target_name, target_path in all_datasets_paths.items():
        if target_name == current_dataset_name:
            continue

        print("\n" + "=" * 70)
        print(f"Testing on unseen dataset: {target_name}")
        print(f"Path: {target_path}")
        print("=" * 70)

        tokenized_texts, x_ml, y_true = _prepare_cross_dataset_ml_matrix(
            target_path,
            w2v_model,
            scaler
        )
        y_true = np.asarray(y_true, dtype=int)
        print(f"Samples: {len(y_true)}")

        ml_tables = []
        for model_name, model in ml_models.items():
            y_pred = np.asarray(model.predict(x_ml), dtype=int)
            metrics = evaluate_classifier(model, x_ml, y_true)
            ml_tables.append(_build_unseen_summary_table(model_name, 'ML', y_true, y_pred, metrics))

        ml_df = pd.concat(ml_tables).sort_index()

        dl_loader = _prepare_cross_dataset_dl_loader(
            tokenized_texts,
            y_true,
            vocab,
            max_len=max_len,
            batch_size=batch_size
        )

        dl_tables = []
        criterion = nn.BCEWithLogitsLoss()

        for model_name, model_entry in dl_models.items():
            trained_model = model_entry['model'] if isinstance(model_entry, dict) else model_entry
            device = next(trained_model.parameters()).device

            dl_metrics = _evaluate_dl(trained_model, dl_loader, criterion, device)
            dl_tables.append(_build_unseen_summary_table(
                model_name,
                'DL',
                np.asarray(dl_metrics['labels'], dtype=int),
                np.asarray(dl_metrics['preds'], dtype=int),
                dl_metrics
            ))

        dl_df = pd.concat(dl_tables).sort_index()

        combined_df = pd.concat([ml_df, dl_df]).sort_index()
        print("\nCombined Unseen-Dataset Summary (ML + DL)")
        display(combined_df)

        all_results[target_name] = {
            'combined_summary': combined_df,
            'ml_summary': ml_df,
            'dl_summary': dl_df
        }

    return all_results


DATASETS = {
    'WELFake': '/content/drive/MyDrive/datasets/WELFake_processed.csv',
    'FakeNewsNet': '/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv',
    'Fake_News_Detection': '/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv',
    'ISOT': '/content/drive/MyDrive/datasets/ISOT_processed.csv',
    'Fake_News_Classification': '/content/drive/MyDrive/datasets/Fake_News_Classification_processed.csv'
}


def new_train_loop_word2vec(dataset_name, datasets_map, **pipeline_kwargs):
    source_path = datasets_map[dataset_name]
    print(f"\nInitialising Word2Vec experiment on: {dataset_name}")

    result_bundle = run_hybrid_pipeline_v2(
        dataset_csv_path=source_path,
        **pipeline_kwargs
    )

    cross_results = cross_dataset_evaluation_word2vec(
        result_bundle=result_bundle,
        current_dataset_name=dataset_name,
        all_datasets_paths=datasets_map
    )

    return result_bundle, cross_results

In [ ]:
# Running pipeline
source_dataset = 'WELFake'

result_bundle, cross_dataset_results = new_train_loop_word2vec(
    dataset_name=source_dataset,
    datasets_map=DATASETS,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.15,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
)

print('\nRanking criterion: validation F1 (primary); test metrics reported across fold models.')

print('\nML Ranked Results (validation-F1 primary):')
display(result_bundle['ml_results'])

if result_bundle['ml_results_cv'] is not None:
    print('\nML CV Summary:')
    display(result_bundle['ml_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['ml_results_test'] is not None:
    print('\nML Selected-Model Test Metrics:')
    display(result_bundle['ml_results_test'].sort_values('f1', ascending=False))

if result_bundle['ml_fold_results'] is not None:
    print('\nML Fold-level Metrics:')
    display(result_bundle['ml_fold_results'])

print('\nDL Ranked Results (validation-F1 primary):')
display(result_bundle['dl_results'])

if result_bundle['dl_results_cv'] is not None:
    print('\nDL CV Summary:')
    display(result_bundle['dl_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['dl_results_test'] is not None:
    print('\nDL Selected-Model Test Metrics:')
    display(result_bundle['dl_results_test'].sort_values('f1', ascending=False))

if result_bundle['dl_fold_results'] is not None:
    print('\nDL Fold-level Metrics:')
    display(result_bundle['dl_fold_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))
print('Selection Policy:', result_bundle.get('selection_policy', 'n/a'))


Initialising Word2Vec experiment on: WELFake
Split summary (stratified):
Train: 54119 | Test: 9551

Word2Vec regime: strict

ML results with 5-fold CV (ranked by validation F1):
                model  cv_f1_mean  cv_f1_std  test_f1_mean  test_f1_std  \
0                 SVC    0.945812   0.002163      0.945176     0.000220   
1             XGBoost    0.924168   0.003094      0.922798     0.001541   
2  LogisticRegression    0.921378   0.002222      0.925204     0.001016   
3        RandomForest    0.887782   0.001846      0.884300     0.000689   
4                 KNN    0.843701   0.002157      0.847952     0.002071   
5          GaussianNB    0.589446   0.014281      0.580033     0.010391   

   selected_fold  holdout_f1  
0              4    0.945325  
1              4    0.922594  
2              4    0.924439  
3              4    0.885288  
4              5    0.849869  
5              2    0.598498  

TextCNN: 5-fold CV on train partition
Epoch 1/10 | Train Loss: 0.1588 | Val L

precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.231935  0.112176  0.151216   
                              Real (1)       0.754800  0.880341  0.812751   
                              Weighted Avg   0.627411  0.693188  0.651577   
                              Overall        0.627411  0.693188  0.651577   
           TextCNNLSTM        Fake (0)       0.229660  0.275272  0.250406   
                              Real (1)       0.750598  0.702578  0.725795   
                              Weighted Avg   0.623678  0.598471  0.609973   
                              Overall        0.623678  0.598471  0.609973   
           TextLSTM           Fake (0)       0.248158  0.101278  0.143848   
                              Real (1)       0.756863  0.901162  0.822733   
                              Weighted Avg   0.632924  0.706281  0.657332   
                              Overall        0.632924  0.706281  0.657332   
ML         GaussianNB         Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.756363  1.000000  0.861283   
                              Weighted Avg   0.572085  0.756363  0.651443   
                              Overall        0.572085  0.756363  0.651443   
           KNN                Fake (0)       0.231738  0.597896  0.334016   
                              Real (1)       0.736226  0.361518  0.484920   
                              Weighted Avg   0.613314  0.419108  0.448154   
                              Overall        0.613314  0.419108  0.448154   
           LogisticRegression Fake (0)       0.222312  0.258362  0.238985   
                              Real (1)       0.747940  0.708873  0.727883   
                              Weighted Avg   0.619878  0.599112  0.608769   
                              Overall        0.619878  0.599112  0.608769   
           RandomForest       Fake (0)       0.237092  0.131154  0.168885   
                              Real (1)       0.755344  0.864060  0.806053   
                              Weighted Avg   0.629079  0.685497  0.650815   
                              Overall        0.629079  0.685497  0.650815   
           SVC                Fake (0)       0.125000  0.001879  0.003702   
                              Real (1)       0.755927  0.995763  0.859426   
                              Weighted Avg   0.602210  0.753617  0.650941   
                              Overall        0.602210  0.753617  0.650941   
           XGBoost            Fake (0)       0.199267  0.081736  0.115923   
                              Real (1)       0.751437  0.894202  0.816627   
                              Weighted Avg   0.616908  0.696255  0.645909   
                              Overall        0.616908  0.696255  0.645909   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.693188  0.489236   
           TextCNNLSTM        Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.598471  0.482372   
           TextLSTM           Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.706281  0.502208   
ML         GaussianNB         Fake (0)       5322.0       NaN      


Testing on unseen dataset: Fake_News_Detection
Path: /content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv
Samples: 38650

Combined Unseen-Dataset Summary (ML + DL)


precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.009903  0.008879  0.009363   
                              Real (1)       0.247717  0.268802  0.257829   
                              Weighted Avg   0.140310  0.151410  0.145611   
                              Overall        0.140310  0.151410  0.145611   
           TextCNNLSTM        Fake (0)       0.024834  0.028300  0.026454   
                              Real (1)       0.095746  0.084741  0.089908   
                              Weighted Avg   0.063719  0.059250  0.061249   
                              Overall        0.063719  0.059250  0.061249   
           TextLSTM           Fake (0)       0.017340  0.020738  0.018887   
                              Real (1)       0.038204  0.032037  0.034850   
                              Weighted Avg   0.028781  0.026934  0.027640   
                              Overall        0.028781  0.026934  0.027640   
ML         GaussianNB         Fake (0)       0.317669  0.519707  0.394315   
                              Real (1)       0.169243  0.080589  0.109186   
                              Weighted Avg   0.236279  0.278913  0.237963   
                              Overall        0.236279  0.278913  0.237963   
           KNN                Fake (0)       0.133990  0.185380  0.155551   
                              Real (1)       0.019243  0.013164  0.015633   
                              Weighted Avg   0.071068  0.090944  0.078826   
                              Overall        0.071068  0.090944  0.078826   
           LogisticRegression Fake (0)       0.042106  0.051730  0.046424   
                              Real (1)       0.037840  0.030716  0.033908   
                              Weighted Avg   0.039767  0.040207  0.039561   
                              Overall        0.039767  0.040207  0.039561   
           RandomForest       Fake (0)       0.028878  0.035919  0.032016   
                              Real (1)       0.006435  0.005143  0.005717   
                              Weighted Avg   0.016571  0.019043  0.017595   
                              Overall        0.016571  0.019043  0.017595   
           SVC                Fake (0)       0.023795  0.029274  0.026252   
                              Real (1)       0.013392  0.010852  0.011989   
                              Weighted Avg   0.018090  0.019172  0.018431   
                              Overall        0.018090  0.019172  0.018431   
           XGBoost            Fake (0)       0.025523  0.031565  0.028225   
                              Real (1)       0.009202  0.007408  0.008208   
                              Weighted Avg   0.016573  0.018318  0.017248   
                              Overall        0.016573  0.018318  0.017248   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.151410  0.013637   
           TextCNNLSTM        Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.059250  0.012354   
           TextLSTM           Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.026934  0.004172   
ML         GaussianNB         Fake (0)      17456.0       NaN      


Testing on unseen dataset: ISOT
Path: /content/drive/MyDrive/datasets/ISOT_processed.csv
Samples: 39098

Combined Unseen-Dataset Summary (ML + DL)


...


DL Selected-Model Test Metrics:


,model,selected_fold,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,4,0.962308,0.943700,0.975069,0.959128,0.994086,0.992300,5
1,TextLSTM,4,0.950162,0.942202,0.948292,0.945237,0.986184,0.981137,7
2,TextCNNLSTM,3,0.950267,0.951111,0.938596,0.944812,0.989559,0.987463,7



DL Fold-level Metrics:


,model,fold,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,val_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,early_stop_epoch,best_epoch
0,TextCNN,1,0.959534,0.956131,0.954573,0.955352,0.993211,0.991084,0.960214,0.959108,0.952909,0.955998,0.993882,0.992122,4,1
1,TextCNN,2,0.957225,0.938301,0.969450,0.953621,0.992671,0.990560,0.960737,0.941728,0.973684,0.957440,0.993033,0.991094,4,1
2,TextCNN,3,0.962860,0.957201,0.961100,0.959146,0.994539,0.993305,0.962098,0.951958,0.965143,0.958505,0.994033,0.992432,4,1
3,TextCNN,4,0.962768,0.947567,0.971690,0.959477,0.993821,0.992110,0.962308,0.943700,0.975069,0.959128,0.994086,0.992300,5,2
4,TextCNN,5,0.961563,0.954297,0.961296,0.957784,0.992705,0.990356,0.959690,0.955458,0.955679,0.955568,0.993575,0.991806,4,1
5,TextCNNLSTM,1,0.949464,0.933771,0.956407,0.944953,0.988398,0.983412,0.949534,0.940503,0.948753,0.944610,0.988671,0.984441,7,4
6,TextCNNLSTM,2,0.947062,0.920333,0.967006,0.943093,0.988286,0.984071,0.947754,0.922230,0.966297,0.943749,0.988454,0.984565,5,2
7,TextCNNLSTM,3,0.952051,0.956730,0.936660,0.946588,0.990897,0.988562,0.950267,0.951111,0.938596,0.944812,0.989559,0.987463,7,4
8,TextCNNLSTM,4,0.943459,0.947895,0.926273,0.936959,0.987447,0.981768,0.943147,0.945450,0.928209,0.936750,0.987466,0.982975,5,2
9,TextCNNLSTM,5,0.944285,0.950042,0.925850,0.937790,0.986828,0.980358,0.946079,0.957345,0.922207,0.939447,0.989393,0.985922,6,3



Word2Vec Regime: strict
Vocab Size: 102061
Selection Policy: best_fold_by_validation_f1


In [ ]:
# Running pipeline
source_dataset = 'FakeNewsNet'

result_bundle, cross_dataset_results = new_train_loop_word2vec(
    dataset_name=source_dataset,
    datasets_map=DATASETS,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.15,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
)

print('\nRanking criterion: validation F1 (primary); test metrics reported across fold models.')

print('\nML Ranked Results (validation-F1 primary):')
display(result_bundle['ml_results'])

if result_bundle['ml_results_cv'] is not None:
    print('\nML CV Summary:')
    display(result_bundle['ml_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['ml_results_test'] is not None:
    print('\nML Selected-Model Test Metrics:')
    display(result_bundle['ml_results_test'].sort_values('f1', ascending=False))

if result_bundle['ml_fold_results'] is not None:
    print('\nML Fold-level Metrics:')
    display(result_bundle['ml_fold_results'])

print('\nDL Ranked Results (validation-F1 primary):')
display(result_bundle['dl_results'])

if result_bundle['dl_results_cv'] is not None:
    print('\nDL CV Summary:')
    display(result_bundle['dl_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['dl_results_test'] is not None:
    print('\nDL Selected-Model Test Metrics:')
    display(result_bundle['dl_results_test'].sort_values('f1', ascending=False))

if result_bundle['dl_fold_results'] is not None:
    print('\nDL Fold-level Metrics:')
    display(result_bundle['dl_fold_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))
print('Selection Policy:', result_bundle.get('selection_policy', 'n/a'))


Initialising Word2Vec experiment on: FakeNewsNet
Split summary (stratified):
Train: 18567 | Test: 3277

Word2Vec regime: strict

ML results with 5-fold CV (ranked by validation F1):
                model  cv_f1_mean  cv_f1_std  test_f1_mean  test_f1_std  \
0             XGBoost    0.891316   0.002352      0.891115     0.001339   
1        RandomForest    0.885947   0.001563      0.892281     0.000671   
2                 KNN    0.879570   0.002507      0.880576     0.002649   
3                 SVC    0.859238   0.004556      0.867382     0.002519   
4  LogisticRegression    0.835376   0.007981      0.848683     0.002186   
5          GaussianNB    0.831142   0.005167      0.842135     0.000567   

   selected_fold  holdout_f1  
0              2    0.889142  
1              2    0.891687  
2              5    0.885584  
3              5    0.871295  
4              5    0.850673  
5              5    0.842965  

TextCNN: 5-fold CV on train partition
Epoch 1/10 | Train Loss: 0.4262 | V

precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.572198  0.180080  0.273946   
                              Real (1)       0.458944  0.837812  0.593032   
                              Weighted Avg   0.520828  0.478420  0.418680   
                              Overall        0.520828  0.478420  0.418680   
           TextCNNLSTM        Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.453589  1.000000  0.624095   
                              Weighted Avg   0.205743  0.453589  0.283083   
                              Overall        0.205743  0.453589  0.283083   
           TextLSTM           Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.453589  1.000000  0.624095   
                              Weighted Avg   0.205743  0.453589  0.283083   
                              Overall        0.205743  0.453589  0.283083   
ML         GaussianNB         Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.452738  0.996572  0.622622   
                              Weighted Avg   0.205357  0.452034  0.282414   
                              Overall        0.205357  0.452034  0.282414   
           KNN                Fake (0)       0.655065  0.096838  0.168733   
                              Real (1)       0.463137  0.938573  0.620225   
                              Weighted Avg   0.568009  0.478640  0.373524   
                              Overall        0.568009  0.478640  0.373524   
           LogisticRegression Fake (0)       0.492497  0.140558  0.218699   
                              Real (1)       0.443628  0.825519  0.577117   
                              Weighted Avg   0.470330  0.451249  0.381273   
                              Overall        0.470330  0.451249  0.381273   
           RandomForest       Fake (0)       0.622323  0.027565  0.052792   
                              Real (1)       0.455472  0.979848  0.621873   
                              Weighted Avg   0.546641  0.459510  0.310921   
                              Overall        0.546641  0.459510  0.310921   
           SVC                Fake (0)       0.528284  0.274073  0.360908   
                              Real (1)       0.446417  0.705194  0.546731   
                              Weighted Avg   0.491150  0.469625  0.445195   
                              Overall        0.491150  0.469625  0.445195   
           XGBoost            Fake (0)       0.528075  0.068123  0.120678   
                              Real (1)       0.452198  0.926662  0.607799   
                              Weighted Avg   0.493658  0.457547  0.341631   
                              Overall        0.493658  0.457547  0.341631   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.478420  0.467493   
           TextCNNLSTM        Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.453589  0.526133   
           TextLSTM           Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.453589  0.494805   
ML         GaussianNB         Fake (0)      34790.0       NaN      


Testing on unseen dataset: Fake_News_Detection
Path: /content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv
Samples: 38650

Combined Unseen-Dataset Summary (ML + DL)


precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.428391  0.194661  0.267686   
                              Real (1)       0.542353  0.786072  0.641855   
                              Weighted Avg   0.490883  0.518965  0.472864   
                              Overall        0.490883  0.518965  0.472864   
           TextCNNLSTM        Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.548357  1.000000  0.708308   
                              Weighted Avg   0.300695  0.548357  0.388406   
                              Overall        0.300695  0.548357  0.388406   
           TextLSTM           Fake (0)       0.000000  0.000000  0.000000   
                              Real (1)       0.548357  1.000000  0.708308   
                              Weighted Avg   0.300695  0.548357  0.388406   
                              Overall        0.300695  0.548357  0.388406   
ML         GaussianNB         Fake (0)       1.000000  0.000057  0.000115   
                              Real (1)       0.548371  1.000000  0.708320   
                              Weighted Avg   0.752346  0.548383  0.388464   
                              Overall        0.752346  0.548383  0.388464   
           KNN                Fake (0)       0.308568  0.068286  0.111825   
                              Real (1)       0.532469  0.873974  0.661760   
                              Weighted Avg   0.431346  0.510091  0.413386   
                              Overall        0.431346  0.510091  0.413386   
           LogisticRegression Fake (0)       0.500203  0.211847  0.297638   
                              Real (1)       0.559843  0.825658  0.667251   
                              Weighted Avg   0.532907  0.548435  0.500318   
                              Overall        0.532907  0.548435  0.500318   
           RandomForest       Fake (0)       0.288494  0.019535  0.036592   
                              Real (1)       0.543210  0.960319  0.693907   
                              Weighted Avg   0.428169  0.535420  0.397036   
                              Overall        0.428169  0.535420  0.397036   
           SVC                Fake (0)       0.481662  0.355866  0.409317   
                              Real (1)       0.563391  0.684581  0.618101   
                              Weighted Avg   0.526479  0.536119  0.523805   
                              Overall        0.526479  0.536119  0.523805   
           XGBoost            Fake (0)       0.466513  0.092977  0.155051   
                              Real (1)       0.549828  0.912428  0.686170   
                              Weighted Avg   0.512200  0.542329  0.446294   
                              Overall        0.512200  0.542329  0.446294   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.518965  0.543718   
           TextCNNLSTM        Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.548357  0.381304   
           TextLSTM           Fake (0)      17456.0       NaN       NaN   
                              Real (1)      21194.0       NaN       NaN   
                              Weighted Avg  38650.0       NaN       NaN   
                              Overall       38650.0  0.548357  0.545253   
ML         GaussianNB         Fake (0)      17456.0       NaN      


Testing on unseen dataset: ISOT
Path: /content/drive/MyDrive/datasets/ISOT_processed.csv
Samples: 39098

Combined Unseen-Dataset Summary (ML + DL)


...


DL Selected-Model Test Metrics:


,model,selected_fold,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,5,0.841318,0.861840,0.941105,0.899730,0.862857,0.942625,4
1,TextLSTM,3,0.756485,0.756485,1.000000,0.861362,0.474219,0.747368,5
2,TextCNNLSTM,3,0.756485,0.756485,1.000000,0.861362,0.500850,0.756798,9



DL Fold-level Metrics:


,model,fold,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,val_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,early_stop_epoch,best_epoch
0,TextCNN,1,0.837103,0.879477,0.909220,0.894101,0.855206,0.938938,0.837656,0.873418,0.918516,0.895399,0.859375,0.941765,5,2
1,TextCNN,2,0.833333,0.862343,0.927732,0.893843,0.855685,0.939572,0.840403,0.866018,0.933441,0.898466,0.861981,0.942768,4,1
2,TextCNN,3,0.833558,0.869228,0.918120,0.893006,0.851257,0.940391,0.840403,0.869894,0.927793,0.897911,0.864250,0.942137,4,1
3,TextCNN,4,0.830057,0.875474,0.903846,0.889434,0.860138,0.943608,0.837656,0.873131,0.918919,0.895440,0.860871,0.942153,5,2
4,TextCNN,5,0.840291,0.855766,0.948718,0.899848,0.863683,0.944199,0.841318,0.861840,0.941105,0.899730,0.862857,0.942625,4,1
5,TextCNNLSTM,1,0.756327,0.756327,1.000000,0.861260,0.500000,0.756327,0.756485,0.756485,1.000000,0.861362,0.500000,0.756485,6,3
6,TextCNNLSTM,2,0.756327,0.756327,1.000000,0.861260,0.500000,0.756327,0.756485,0.756485,1.000000,0.861362,0.500000,0.756485,4,1
7,TextCNNLSTM,3,0.756531,0.756531,1.000000,0.861392,0.498932,0.756138,0.756485,0.756485,1.000000,0.861362,0.500850,0.756798,9,6
8,TextCNNLSTM,4,0.756262,0.756262,1.000000,0.861218,0.501676,0.756880,0.756485,0.756485,1.000000,0.861362,0.500670,0.756731,10,10
9,TextCNNLSTM,5,0.756262,0.756262,1.000000,0.861218,0.497449,0.755335,0.756485,0.756485,1.000000,0.861362,0.495565,0.754891,4,1



Word2Vec Regime: strict
Vocab Size: 8703
Selection Policy: best_fold_by_validation_f1


In [22]:
# Running pipeline
source_dataset = 'Fake_News_Detection'

result_bundle, cross_dataset_results = new_train_loop_word2vec(
    dataset_name=source_dataset,
    datasets_map=DATASETS,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.15,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
)

print('\nRanking criterion: validation F1 (primary); test metrics reported across fold models.')

print('\nML Ranked Results (validation-F1 primary):')
display(result_bundle['ml_results'])

if result_bundle['ml_results_cv'] is not None:
    print('\nML CV Summary:')
    display(result_bundle['ml_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['ml_results_test'] is not None:
    print('\nML Selected-Model Test Metrics:')
    display(result_bundle['ml_results_test'].sort_values('f1', ascending=False))

if result_bundle['ml_fold_results'] is not None:
    print('\nML Fold-level Metrics:')
    display(result_bundle['ml_fold_results'])

print('\nDL Ranked Results (validation-F1 primary):')
display(result_bundle['dl_results'])

if result_bundle['dl_results_cv'] is not None:
    print('\nDL CV Summary:')
    display(result_bundle['dl_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['dl_results_test'] is not None:
    print('\nDL Selected-Model Test Metrics:')
    display(result_bundle['dl_results_test'].sort_values('f1', ascending=False))

if result_bundle['dl_fold_results'] is not None:
    print('\nDL Fold-level Metrics:')
    display(result_bundle['dl_fold_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))
print('Selection Policy:', result_bundle.get('selection_policy', 'n/a'))


Initialising Word2Vec experiment on: Fake_News_Detection
Split summary (stratified):
Train: 32852 | Test: 5798

Word2Vec regime: strict

ML results with 5-fold CV (ranked by validation F1):
                model  cv_f1_mean  cv_f1_std  test_f1_mean  test_f1_std  \
0                 SVC    0.987982   0.000719      0.987412     0.000389   
1  LogisticRegression    0.985550   0.001805      0.984556     0.000456   
2             XGBoost    0.977669   0.001974      0.977564     0.000806   
3        RandomForest    0.959332   0.001988      0.957740     0.000367   
4                 KNN    0.954495   0.001554      0.951369     0.001457   
5          GaussianNB    0.909958   0.001678      0.905507     0.000261   

   selected_fold  holdout_f1  
0              4    0.986696  
1              4    0.984137  
2              2    0.978471  
3              5    0.958404  
4              2    0.951573  
5              2    0.905780  

TextCNN: 5-fold CV on train partition
Epoch 1/10 | Train Loss: 0.

precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.227558  0.229376  0.228463   
                              Real (1)       0.062653  0.062050  0.062350   
                              Weighted Avg   0.152759  0.153479  0.153116   
                              Overall        0.152759  0.153479  0.153116   
           TextCNNLSTM        Fake (0)       0.231658  0.233343  0.232498   
                              Real (1)       0.068292  0.067694  0.067992   
                              Weighted Avg   0.157557  0.158206  0.157880   
                              Overall        0.157557  0.158206  0.157880   
           TextLSTM           Fake (0)       0.241169  0.246479  0.243795   
                              Real (1)       0.067546  0.065755  0.066639   
                              Weighted Avg   0.162416  0.164504  0.163439   
                              Overall        0.162416  0.164504  0.163439   
ML         GaussianNB         Fake (0)       0.329309  0.330066  0.329687   
                              Real (1)       0.190729  0.190201  0.190465   
                              Weighted Avg   0.266451  0.266625  0.266537   
                              Overall        0.266451  0.266625  0.266537   
           KNN                Fake (0)       0.229035  0.219891  0.224370   
                              Real (1)       0.103373  0.108345  0.105801   
                              Weighted Avg   0.172036  0.169295  0.170588   
                              Overall        0.172036  0.169295  0.170588   
           LogisticRegression Fake (0)       0.228346  0.229376  0.228860   
                              Real (1)       0.066602  0.066240  0.066420   
                              Weighted Avg   0.154980  0.155379  0.155179   
                              Overall        0.154980  0.155379  0.155179   
           RandomForest       Fake (0)       0.238220  0.233515  0.235844   
                              Real (1)       0.098116  0.100450  0.099269   
                              Weighted Avg   0.174670  0.173158  0.173895   
                              Overall        0.174670  0.173158  0.173895   
           SVC                Fake (0)       0.225858  0.220437  0.223114   
                              Real (1)       0.087296  0.089820  0.088540   
                              Weighted Avg   0.163008  0.161191  0.162073   
                              Overall        0.163008  0.161191  0.162073   
           XGBoost            Fake (0)       0.216939  0.213510  0.215211   
                              Real (1)       0.070268  0.071607  0.070931   
                              Weighted Avg   0.150411  0.149144  0.149767   
                              Overall        0.150411  0.149144  0.149767   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.153479  0.060036   
           TextCNNLSTM        Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.158206  0.080692   
           TextLSTM           Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.164504  0.097607   
ML         GaussianNB         Fake (0)      34790.0       NaN      


Testing on unseen dataset: FakeNewsNet
Path: /content/drive/MyDrive/datasets/FakeNewsNet_processed.csv
Samples: 21844

Combined Unseen-Dataset Summary (ML + DL)


precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.248845  0.870537  0.387051   
                              Real (1)       0.786423  0.153553  0.256937   
                              Weighted Avg   0.655449  0.328237  0.288638   
                              Overall        0.655449  0.328237  0.288638   
           TextCNNLSTM        Fake (0)       0.244237  0.800263  0.374253   
                              Real (1)       0.758738  0.202336  0.319476   
                              Weighted Avg   0.633387  0.348013  0.332822   
                              Overall        0.633387  0.348013  0.332822   
           TextLSTM           Fake (0)       0.249656  0.784667  0.378793   
                              Real (1)       0.776041  0.240346  0.367023   
                              Weighted Avg   0.647794  0.372963  0.369890   
                              Overall        0.647794  0.372963  0.369890   
ML         GaussianNB         Fake (0)       0.181613  0.060504  0.090768   
                              Real (1)       0.750884  0.912178  0.823709   
                              Weighted Avg   0.612189  0.704679  0.645138   
                              Overall        0.612189  0.704679  0.645138   
           KNN                Fake (0)       0.243557  0.843480  0.377973   
                              Real (1)       0.755933  0.156155  0.258841   
                              Weighted Avg   0.631100  0.323613  0.287866   
                              Overall        0.631100  0.323613  0.287866   
           LogisticRegression Fake (0)       0.249834  0.778091  0.378225   
                              Real (1)       0.775859  0.247428  0.375201   
                              Weighted Avg   0.647700  0.376717  0.375938   
                              Overall        0.647700  0.376717  0.375938   
           RandomForest       Fake (0)       0.240459  0.719842  0.360497   
                              Real (1)       0.747801  0.267583  0.394134   
                              Weighted Avg   0.624194  0.377770  0.385939   
                              Overall        0.624194  0.377770  0.385939   
           SVC                Fake (0)       0.228930  0.250094  0.239045   
                              Real (1)       0.751029  0.728665  0.739678   
                              Weighted Avg   0.623827  0.612067  0.617705   
                              Overall        0.623827  0.612067  0.617705   
           XGBoost            Fake (0)       0.244735  0.834085  0.378431   
                              Real (1)       0.761738  0.170863  0.279118   
                              Weighted Avg   0.635777  0.332448  0.303314   
                              Overall        0.635777  0.332448  0.303314   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.328237  0.513345   
           TextCNNLSTM        Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.348013  0.500934   
           TextLSTM           Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.372963  0.517661   
ML         GaussianNB         Fake (0)       5322.0       NaN      


Testing on unseen dataset: ISOT
Path: /content/drive/MyDrive/datasets/ISOT_processed.csv
Samples: 39098

Combined Unseen-Dataset Summary (ML + DL)


...


DL Selected-Model Test Metrics:


,model,selected_fold,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,1,0.988962,0.985354,0.994652,0.989981,0.999187,0.999298,5
1,TextLSTM,1,0.986547,0.986813,0.988676,0.987744,0.998245,0.998481,7
2,TextCNNLSTM,4,0.984305,0.981898,0.989619,0.985743,0.998769,0.998933,7



DL Fold-level Metrics:


,model,fold,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,val_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,early_stop_epoch,best_epoch
0,TextCNN,1,0.992086,0.988984,0.996669,0.992812,0.999653,0.999715,0.988962,0.985354,0.994652,0.989981,0.999187,0.999298,5,2
1,TextCNN,2,0.991173,0.991133,0.992784,0.991958,0.999449,0.999529,0.989307,0.992106,0.988361,0.990230,0.999373,0.999488,6,3
2,TextCNN,3,0.989193,0.989739,0.990563,0.990151,0.999170,0.999302,0.988444,0.989308,0.989619,0.989464,0.999273,0.999396,4,1
3,TextCNN,4,0.987519,0.983256,0.994172,0.988683,0.999019,0.998690,0.987065,0.981688,0.994967,0.988283,0.999382,0.999487,4,1
4,TextCNN,5,0.991629,0.991958,0.992784,0.992371,0.999368,0.999401,0.987927,0.992086,0.985845,0.988956,0.999380,0.999489,5,2
5,TextCNNLSTM,1,0.986608,0.988873,0.986678,0.987774,0.998367,0.998564,0.980683,0.980269,0.984586,0.982423,0.997631,0.997742,8,5
6,TextCNNLSTM,2,0.984477,0.990474,0.981127,0.985778,0.998390,0.998650,0.984477,0.991409,0.980182,0.985764,0.997736,0.998068,8,5
7,TextCNNLSTM,3,0.984018,0.980759,0.990286,0.985499,0.998104,0.998456,0.984477,0.983714,0.988047,0.985876,0.997054,0.996923,5,2
8,TextCNNLSTM,4,0.987823,0.987545,0.990286,0.988914,0.998672,0.998796,0.984305,0.981898,0.989619,0.985743,0.998769,0.998933,7,4
9,TextCNNLSTM,5,0.987823,0.988898,0.988898,0.988898,0.998681,0.998869,0.982063,0.982429,0.984901,0.983663,0.997626,0.997943,10,7



Word2Vec Regime: strict
Vocab Size: 56570
Selection Policy: best_fold_by_validation_f1


In [ ]:
# Running pipeline
source_dataset = 'Fake_News_Classification'

result_bundle, cross_dataset_results = new_train_loop_word2vec(
    dataset_name=source_dataset,
    datasets_map=DATASETS,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.15,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
)

print('\nRanking criterion: validation F1 (primary); test metrics reported across fold models.')

print('\nML Ranked Results (validation-F1 primary):')
display(result_bundle['ml_results'])

if result_bundle['ml_results_cv'] is not None:
    print('\nML CV Summary:')
    display(result_bundle['ml_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['ml_results_test'] is not None:
    print('\nML Selected-Model Test Metrics:')
    display(result_bundle['ml_results_test'].sort_values('f1', ascending=False))

if result_bundle['ml_fold_results'] is not None:
    print('\nML Fold-level Metrics:')
    display(result_bundle['ml_fold_results'])

print('\nDL Ranked Results (validation-F1 primary):')
display(result_bundle['dl_results'])

if result_bundle['dl_results_cv'] is not None:
    print('\nDL CV Summary:')
    display(result_bundle['dl_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['dl_results_test'] is not None:
    print('\nDL Selected-Model Test Metrics:')
    display(result_bundle['dl_results_test'].sort_values('f1', ascending=False))

if result_bundle['dl_fold_results'] is not None:
    print('\nDL Fold-level Metrics:')
    display(result_bundle['dl_fold_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))
print('Selection Policy:', result_bundle.get('selection_policy', 'n/a'))


Initialising Word2Vec experiment on: Fake_News_Classification
Split summary (stratified):
Train: 34493 | Test: 6087

Word2Vec regime: strict

ML results with 5-fold CV (ranked by validation F1):
                model  cv_f1_mean  cv_f1_std  test_f1_mean  test_f1_std  \
0                 SVC    0.976100   0.001160      0.977535     0.000425   
1  LogisticRegression    0.969239   0.001790      0.972326     0.000354   
2             XGBoost    0.965302   0.001290      0.966533     0.000675   
3        RandomForest    0.946893   0.001851      0.949227     0.000792   
4                 KNN    0.939668   0.002107      0.939537     0.001295   
5          GaussianNB    0.903524   0.003869      0.904968     0.001077   

   selected_fold  holdout_f1  
0              1    0.978287  
1              4    0.971655  
2              1    0.967193  
3              4    0.948932  
4              4    0.939820  
5              1    0.905137  

TextCNN: 5-fold CV on train partition
Epoch 1/10 | Train Los

precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.318032  0.367031  0.340779   
                              Real (1)       0.063733  0.051904  0.057214   
                              Weighted Avg   0.202685  0.224093  0.212157   
                              Overall        0.202685  0.224093  0.212157   
           TextCNNLSTM        Fake (0)       0.316202  0.365536  0.339084   
                              Real (1)       0.058801  0.047749  0.052702   
                              Weighted Avg   0.199448  0.221392  0.209184   
                              Overall        0.199448  0.221392  0.209184   
           TextLSTM           Fake (0)       0.309372  0.351452  0.329072   
                              Real (1)       0.065637  0.054882  0.059780   
                              Weighted Avg   0.198816  0.216931  0.206924   
                              Overall        0.198816  0.216931  0.206924   
ML         GaussianNB         Fake (0)       0.316278  0.321242  0.318741   
                              Real (1)       0.166584  0.163435  0.164995   
                              Weighted Avg   0.248379  0.249662  0.249003   
                              Overall        0.248379  0.249662  0.249003   
           KNN                Fake (0)       0.236089  0.225496  0.230671   
                              Real (1)       0.114845  0.121053  0.117867   
                              Weighted Avg   0.181094  0.178122  0.179504   
                              Overall        0.181094  0.178122  0.179504   
           LogisticRegression Fake (0)       0.222268  0.215234  0.218695   
                              Real (1)       0.089357  0.092763  0.091028   
                              Weighted Avg   0.161981  0.159683  0.160787   
                              Overall        0.161981  0.159683  0.160787   
           RandomForest       Fake (0)       0.241881  0.240414  0.241145   
                              Real (1)       0.091609  0.092278  0.091943   
                              Weighted Avg   0.173719  0.173221  0.173469   
                              Overall        0.173719  0.173221  0.173469   
           SVC                Fake (0)       0.226287  0.230612  0.228429   
                              Real (1)       0.051320  0.050139  0.050722   
                              Weighted Avg   0.146924  0.148751  0.147823   
                              Overall        0.146924  0.148751  0.147823   
           XGBoost            Fake (0)       0.219642  0.216384  0.218001   
                              Real (1)       0.072595  0.073892  0.073238   
                              Weighted Avg   0.152943  0.151751  0.152338   
                              Overall        0.152943  0.151751  0.152338   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.224093  0.090555   
           TextCNNLSTM        Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.221392  0.134427   
           TextLSTM           Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.216931  0.126186   
ML         GaussianNB         Fake (0)      34790.0       NaN      


Testing on unseen dataset: FakeNewsNet
Path: /content/drive/MyDrive/datasets/FakeNewsNet_processed.csv
Samples: 21844

Combined Unseen-Dataset Summary (ML + DL)


precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.243448  0.946073  0.387248   
                              Real (1)       0.753012  0.052960  0.098960   
                              Weighted Avg   0.628864  0.270555  0.169197   
                              Overall        0.628864  0.270555  0.169197   
           TextCNNLSTM        Fake (0)       0.245647  0.898722  0.385835   
                              Real (1)       0.772861  0.111004  0.194125   
                              Weighted Avg   0.644413  0.302921  0.240833   
                              Overall        0.644413  0.302921  0.240833   
           TextLSTM           Fake (0)       0.244114  0.942879  0.387820   
                              Real (1)       0.763975  0.059557  0.110500   
                              Weighted Avg   0.637318  0.274767  0.178065   
                              Overall        0.637318  0.274767  0.178065   
ML         GaussianNB         Fake (0)       0.212117  0.287486  0.244116   
                              Real (1)       0.740824  0.656034  0.695856   
                              Weighted Avg   0.612012  0.566242  0.585796   
                              Overall        0.612012  0.566242  0.585796   
           KNN                Fake (0)       0.243517  0.827508  0.376298   
                              Real (1)       0.755786  0.171953  0.280164   
                              Weighted Avg   0.630978  0.331670  0.303585   
                              Overall        0.630978  0.331670  0.303585   
           LogisticRegression Fake (0)       0.252294  0.795566  0.383098   
                              Real (1)       0.785065  0.240528  0.368236   
                              Weighted Avg   0.655263  0.375755  0.371857   
                              Overall        0.655263  0.375755  0.371857   
           RandomForest       Fake (0)       0.249833  0.771890  0.377487   
                              Real (1)       0.775227  0.253420  0.381973   
                              Weighted Avg   0.647222  0.379738  0.380880   
                              Overall        0.647222  0.379738  0.380880   
           SVC                Fake (0)       0.244403  0.992860  0.392250   
                              Real (1)       0.830357  0.011258  0.022214   
                              Weighted Avg   0.687597  0.250412  0.112369   
                              Overall        0.687597  0.250412  0.112369   
           XGBoost            Fake (0)       0.250472  0.872980  0.389259   
                              Real (1)       0.794841  0.158516  0.264319   
                              Weighted Avg   0.662212  0.332586  0.294759   
                              Overall        0.662212  0.332586  0.294759   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.270555  0.498428   
           TextCNNLSTM        Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.302921  0.513160   
           TextLSTM           Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.274767  0.520562   
ML         GaussianNB         Fake (0)       5322.0       NaN      


Testing on unseen dataset: Fake_News_Detection
Path: /content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv
Samples: 38650

Combined Unseen-Dataset Summary (ML + DL)


...


DL Selected-Model Test Metrics:


,model,selected_fold,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNN,1,0.990307,0.995397,0.986618,0.990988,0.999421,0.999568,5
1,TextCNNLSTM,5,0.988829,0.995080,0.984185,0.989602,0.997529,0.998393,10
2,TextLSTM,5,0.986857,0.992327,0.983273,0.987779,0.997759,0.998380,9



DL Fold-level Metrics:


,model,fold,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,val_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,early_stop_epoch,best_epoch
0,TextCNN,1,0.987969,0.993232,0.984438,0.988816,0.999301,0.999429,0.990307,0.995397,0.986618,0.990988,0.999421,0.999568,5,2
1,TextCNN,2,0.985795,0.992669,0.980950,0.986775,0.998945,0.999127,0.990800,0.996620,0.986314,0.991440,0.999341,0.999490,4,1
2,TextCNN,3,0.985070,0.992659,0.979608,0.986090,0.999042,0.999224,0.989157,0.995692,0.984185,0.989905,0.999314,0.999474,4,1
3,TextCNN,4,0.986518,0.995906,0.979072,0.987417,0.998947,0.998207,0.989157,0.996915,0.982968,0.989893,0.999266,0.999455,4,1
4,TextCNN,5,0.987098,0.990032,0.986048,0.988036,0.999311,0.999439,0.991129,0.994798,0.988747,0.991763,0.999322,0.999538,5,2
5,TextCNNLSTM,1,0.983186,0.989163,0.979608,0.984362,0.996584,0.997242,0.988829,0.995080,0.984185,0.989602,0.997097,0.997995,7,4
6,TextCNNLSTM,2,0.979562,0.988290,0.973705,0.980943,0.995964,0.997096,0.985543,0.991703,0.981448,0.986548,0.997054,0.998024,9,6
7,TextCNNLSTM,3,0.977243,0.978552,0.979340,0.978946,0.996654,0.997164,0.982586,0.982121,0.985706,0.983910,0.997557,0.998295,8,5
8,TextCNNLSTM,4,0.984343,0.996433,0.974510,0.985350,0.995908,0.997424,0.985214,0.997201,0.975365,0.986162,0.996111,0.997602,8,5
9,TextCNNLSTM,5,0.985938,0.992137,0.981755,0.986918,0.997660,0.998254,0.988829,0.995080,0.984185,0.989602,0.997529,0.998393,10,7



Word2Vec Regime: strict
Vocab Size: 61971
Selection Policy: best_fold_by_validation_f1


In [ ]:
# Running pipeline
source_dataset = 'ISOT'

result_bundle, cross_dataset_results = new_train_loop_word2vec(
    dataset_name=source_dataset,
    datasets_map=DATASETS,
    vector_size=300,
    window=5,
    min_count=2,
    w2v_epochs=10,
    max_len=200,
    test_size=0.15,
    val_size=0.2,
    dl_epochs=10,
    dl_batch_size=32,
    dl_lr=1e-3,
    dl_patience=3,
    seed=42,
    transductive_w2v=False,
    enable_cv=True,
    cv_folds=5
)

print('\nRanking criterion: validation F1 (primary); test metrics reported across fold models.')

print('\nML Ranked Results (validation-F1 primary):')
display(result_bundle['ml_results'])

if result_bundle['ml_results_cv'] is not None:
    print('\nML CV Summary:')
    display(result_bundle['ml_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['ml_results_test'] is not None:
    print('\nML Selected-Model Test Metrics:')
    display(result_bundle['ml_results_test'].sort_values('f1', ascending=False))

if result_bundle['ml_fold_results'] is not None:
    print('\nML Fold-level Metrics:')
    display(result_bundle['ml_fold_results'])

print('\nDL Ranked Results (validation-F1 primary):')
display(result_bundle['dl_results'])

if result_bundle['dl_results_cv'] is not None:
    print('\nDL CV Summary:')
    display(result_bundle['dl_results_cv'].sort_values('cv_f1_mean', ascending=False))

if result_bundle['dl_results_test'] is not None:
    print('\nDL Selected-Model Test Metrics:')
    display(result_bundle['dl_results_test'].sort_values('f1', ascending=False))

if result_bundle['dl_fold_results'] is not None:
    print('\nDL Fold-level Metrics:')
    display(result_bundle['dl_fold_results'])

print('\nWord2Vec Regime:', result_bundle['word2vec_regime'])
print('Vocab Size:', len(result_bundle['vocab']))
print('Selection Policy:', result_bundle.get('selection_policy', 'n/a'))


Initialising Word2Vec experiment on: ISOT
Split summary (stratified):
Train: 33233 | Test: 5865

Word2Vec regime: strict

ML results with 5-fold CV (ranked by validation F1):
                model  cv_f1_mean  cv_f1_std  test_f1_mean  test_f1_std  \
0                 SVC    0.988891   0.001321      0.986655     0.000365   
1  LogisticRegression    0.986121   0.001313      0.985708     0.000670   
2             XGBoost    0.977235   0.001688      0.972808     0.000941   
3        RandomForest    0.955766   0.001788      0.951065     0.000926   
4                 KNN    0.946020   0.002741      0.941364     0.001447   
5          GaussianNB    0.901225   0.002968      0.901859     0.000763   

   selected_fold  holdout_f1  
0              4    0.986542  
1              1    0.984518  
2              4    0.973531  
3              4    0.952129  
4              5    0.944126  
5              5    0.902488  

TextCNN: 5-fold CV on train partition
Epoch 1/10 | Train Loss: 0.0284 | Val Loss

precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.988874  0.618252  0.760828   
                              Real (1)       0.683175  0.991620  0.808994   
                              Weighted Avg   0.850212  0.787608  0.782676   
                              Overall        0.850212  0.787608  0.782676   
           TextCNNLSTM        Fake (0)       0.970311  0.648204  0.777205   
                              Real (1)       0.697272  0.976108  0.813459   
                              Weighted Avg   0.846464  0.796937  0.793649   
                              Overall        0.846464  0.796937  0.793649   
           TextLSTM           Fake (0)       0.984498  0.627939  0.766795   
                              Real (1)       0.687946  0.988089  0.811143   
                              Weighted Avg   0.849985  0.791299  0.786911   
                              Overall        0.849985  0.791299  0.786911   
ML         GaussianNB         Fake (0)       0.834574  0.673728  0.745575   
                              Real (1)       0.681017  0.839127  0.751850   
                              Weighted Avg   0.764922  0.748751  0.748421   
                              Overall        0.764922  0.748751  0.748421   
           KNN                Fake (0)       0.895736  0.771687  0.829097   
                              Real (1)       0.764289  0.891794  0.823133   
                              Weighted Avg   0.836113  0.826166  0.826392   
                              Overall        0.836113  0.826166  0.826392   
           LogisticRegression Fake (0)       0.938650  0.752458  0.835304   
                              Real (1)       0.759314  0.940755  0.840352   
                              Weighted Avg   0.857305  0.837867  0.837594   
                              Overall        0.857305  0.837867  0.837594   
           RandomForest       Fake (0)       0.915964  0.761943  0.831885   
                              Real (1)       0.761532  0.915789  0.831567   
                              Weighted Avg   0.845915  0.831726  0.831741   
                              Overall        0.845915  0.831726  0.831741   
           SVC                Fake (0)       0.950459  0.759356  0.844228   
                              Real (1)       0.766634  0.952320  0.849448   
                              Weighted Avg   0.867078  0.846882  0.846596   
                              Overall        0.867078  0.846882  0.846596   
           XGBoost            Fake (0)       0.938377  0.768612  0.845053   
                              Real (1)       0.771138  0.939197  0.846910   
                              Weighted Avg   0.862519  0.845987  0.845895   
                              Overall        0.862519  0.845987  0.845895   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.787608  0.927262   
           TextCNNLSTM        Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.796937  0.897972   
           TextLSTM           Fake (0)      34790.0       NaN       NaN   
                              Real (1)      28880.0       NaN       NaN   
                              Weighted Avg  63670.0       NaN       NaN   
                              Overall       63670.0  0.791299  0.855960   
ML         GaussianNB         Fake (0)      34790.0       NaN      


Testing on unseen dataset: FakeNewsNet
Path: /content/drive/MyDrive/datasets/FakeNewsNet_processed.csv
Samples: 21844

Combined Unseen-Dataset Summary (ML + DL)


precision    recall        f1  \
model_type model              sub_row                                       
DL         TextCNN            Fake (0)       0.303797  0.004510  0.008887   
                              Real (1)       0.756582  0.996671  0.860188   
                              Weighted Avg   0.646267  0.754944  0.652780   
                              Overall        0.646267  0.754944  0.652780   
           TextCNNLSTM        Fake (0)       0.202265  0.023487  0.042088   
                              Real (1)       0.755159  0.970161  0.849264   
                              Weighted Avg   0.620454  0.739517  0.652606   
                              Overall        0.620454  0.739517  0.652606   
           TextLSTM           Fake (0)       0.179487  0.002631  0.005185   
                              Real (1)       0.756133  0.996126  0.859695   
                              Weighted Avg   0.615641  0.754074  0.651505   
                              Overall        0.615641  0.754074  0.651505   
ML         GaussianNB         Fake (0)       0.259901  0.723788  0.382465   
                              Real (1)       0.790688  0.336097  0.471693   
                              Weighted Avg   0.661369  0.430553  0.449954   
                              Overall        0.661369  0.430553  0.449954   
           KNN                Fake (0)       0.242512  0.156708  0.190389   
                              Real (1)       0.756153  0.842331  0.796919   
                              Weighted Avg   0.631011  0.675288  0.649146   
                              Overall        0.631011  0.675288  0.649146   
           LogisticRegression Fake (0)       0.210843  0.177565  0.192778   
                              Real (1)       0.747898  0.785922  0.766438   
                              Weighted Avg   0.617052  0.637704  0.626674   
                              Overall        0.617052  0.637704  0.626674   
           RandomForest       Fake (0)       0.237832  0.129463  0.167660   
                              Real (1)       0.755476  0.866360  0.807127   
                              Weighted Avg   0.629359  0.686825  0.651330   
                              Overall        0.629359  0.686825  0.651330   
           SVC                Fake (0)       0.214286  0.001691  0.003356   
                              Real (1)       0.756307  0.998003  0.860505   
                              Weighted Avg   0.624251  0.755265  0.651672   
                              Overall        0.624251  0.755265  0.651672   
           XGBoost            Fake (0)       0.213842  0.084179  0.120804   
                              Real (1)       0.753203  0.900315  0.820214   
                              Weighted Avg   0.621795  0.701474  0.649812   
                              Overall        0.621795  0.701474  0.649812   

                                            support  accuracy   roc_auc  \
model_type model              sub_row                                     
DL         TextCNN            Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.754944  0.509848   
           TextCNNLSTM        Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.739517  0.493508   
           TextLSTM           Fake (0)       5322.0       NaN       NaN   
                              Real (1)      16522.0       NaN       NaN   
                              Weighted Avg  21844.0       NaN       NaN   
                              Overall       21844.0  0.754074  0.506498   
ML         GaussianNB         Fake (0)       5322.0       NaN      


Testing on unseen dataset: Fake_News_Detection
Path: /content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv
Samples: 38650

Combined Unseen-Dataset Summary (ML + DL)


...


DL Selected-Model Test Metrics:


,model,selected_fold,accuracy,precision,recall,f1,roc_auc,pr_auc,early_stop_epoch
0,TextCNNLSTM,1,0.998465,0.998138,0.998510,0.998324,0.999937,0.999923,10
1,TextCNN,1,0.998295,0.998509,0.997765,0.998137,0.999989,0.999986,4
2,TextLSTM,1,0.998124,0.997766,0.998138,0.997952,0.999767,0.999720,8



DL Fold-level Metrics:


,model,fold,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,val_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,early_stop_epoch,best_epoch
0,TextCNN,1,0.998646,0.998685,0.998357,0.998521,0.999965,0.999960,0.998295,0.998509,0.997765,0.998137,0.999989,0.999986,4,1
1,TextCNN,2,0.998044,0.998028,0.997700,0.997864,0.999959,0.999951,0.998124,0.997396,0.998510,0.997953,0.999978,0.999973,4,1
2,TextCNN,3,0.998496,0.998357,0.998357,0.998357,0.999991,0.999989,0.998295,0.997767,0.998510,0.998138,0.999986,0.999984,6,3
3,TextCNN,4,0.998495,0.998357,0.998357,0.998357,0.999957,0.999953,0.998636,0.997399,0.999628,0.998512,0.999985,0.999982,5,2
4,TextCNN,5,0.998495,0.997050,0.999671,0.998359,0.999964,0.999957,0.997954,0.997025,0.998510,0.997767,0.999981,0.999978,4,1
5,TextCNNLSTM,1,0.999097,0.999671,0.998357,0.999013,0.999912,0.999876,0.998465,0.998138,0.998510,0.998324,0.999937,0.999923,10,7
6,TextCNNLSTM,2,0.997593,0.996068,0.998686,0.997375,0.999886,0.999879,0.997613,0.995547,0.999255,0.997398,0.999833,0.999794,10,7
7,TextCNNLSTM,3,0.998195,0.999013,0.997043,0.998027,0.999691,0.999808,0.997442,0.997763,0.996648,0.997205,0.999839,0.999829,10,10
8,TextCNNLSTM,4,0.998345,0.998356,0.998028,0.998192,0.999900,0.999886,0.997954,0.998136,0.997393,0.997765,0.999883,0.999877,10,7
9,TextCNNLSTM,5,0.998646,0.997377,0.999671,0.998523,0.999512,0.999140,0.998124,0.997766,0.998138,0.997952,0.999900,0.999874,9,6



Word2Vec Regime: strict
Vocab Size: 56576
Selection Policy: best_fold_by_validation_f1
